# PCB Phase 3 — source specialist pilot

Attach the same version-2 PCB dataset. Select **GPU T4** and enable Internet for dependency setup. No dataset re-upload is needed. Start in a fresh kernel when switching notebooks.

This run evaluates the existing source baseline, then trains YOLOv8s from COCO weights for **3 epochs on 953 training views**. Validation uses **120 separate full-board images**, at 960 pixels, batch 4. Augmentations are disabled for this controlled pilot. Full precision and zero loader workers simplify recovery. This is a runtime and learning pilot, not the final accuracy experiment.

The run saves complete checkpoint ZIPs after each epoch. Download the ZIP before ending the session. Rerunning resumes local progress; an attached ZIP or expanded checkpoint can restore another session. Resume requires matching package, code, recipe and dependency versions.


In [ ]:
import sys, subprocess, importlib.metadata as meta
from pathlib import Path
from packaging.requirements import Requirement
# Preserve Kaggle's CUDA stack and NumPy ABI while resolving other dependencies.
constraints = Path('/kaggle/working/pcb_runtime_constraints.txt')
constraints.write_text(''.join(f'{name}=={meta.version(name)}\n' for name in ['torch', 'torchvision', 'numpy']))
def satisfied():
    try:
        if meta.version('ultralytics') != '8.4.142': return False
        for text in meta.requires('ultralytics'):
            req = Requirement(text)
            if req.marker and not req.marker.evaluate({'extra': ''}): continue
            if meta.version(req.name) not in req.specifier: return False
        return True
    except meta.PackageNotFoundError:
        return False
if not satisfied():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.142', '--constraint', str(constraints)], check=True)
assert satisfied(), 'Ultralytics dependencies are incomplete'
import torch, torchvision


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, zipfile, os
WORK = Path('/kaggle/working/pcb_phase2')
if not WORK.exists():
    INPUT = Path('/kaggle/input')
    records = list(INPUT.rglob('package_hashes.json'))
    archives = list(INPUT.rglob('pcb_revamp_phase2.zip'))
    if len(records) == 1:
        shutil.copytree(records[0].parent, WORK)
    elif not records and len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as z:
            names = z.namelist()
            if len(names) != len(set(names)): raise ValueError('Duplicate archive entries')
            for info in z.infolist():
                p = PurePosixPath(info.filename)
                if p.is_absolute() or '..' in p.parts or '\\' in info.filename or ((info.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('Unsafe archive path')
            if z.testzip() is not None: raise ValueError('Damaged archive')
            z.extractall(WORK)
    else:
        raise RuntimeError('Attach exactly one Phase 2 package (expanded dataset or ZIP).')

def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
EXPECTED_MANIFEST = '8687e5c1a3b91d3953abe7b10ebe8c793adfdb46f7280be9b8545f9f7ceb1419'
assert file_hash(WORK/'package_hashes.json') == EXPECTED_MANIFEST, 'Wrong package version'
record = json.loads((WORK/'package_hashes.json').read_text())
for name, expected in record['sha256'].items():
    path = (WORK/name).resolve()
    assert path.is_relative_to(WORK.resolve()), 'Unsafe package path'
    assert file_hash(path) == expected, 'Changed asset: '+name
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/pcb_yolo_settings'
os.environ['WANDB_MODE'] = 'disabled'
EMBEDDED_TOOLS = {'__init__.py': {'base64': 'IiIiUGhhc2VkLCBub24tZGVzdHJ1Y3RpdmUgUENCIG1vZGVsIHJldmFtcCB0b29saW5nLiIiIgo=', 'sha256': '35648be55b79bf5b9573ee55b6bd27e7e3c3942a17836c70fc99e29a3f863576'}, 'runtime.py': {'base64': 'IiIiUG9ydGFibGUsIG9mZmxpbmUgcGFja2FnZSB2ZXJpZmljYXRpb24gYW5kIHdyaXRhYmxlIGRhdGFzZXQgY29uZmlndXJhdGlvbi4iIiIKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmltcG9ydCB6aXBmaWxlCgpOQU1FUyA9IFsibWlzc2luZ19ob2xlIiwgIm1vdXNlX2JpdGUiLCAib3Blbl9jaXJjdWl0IiwgInNob3J0IiwgInNwdXIiLCAic3B1cmlvdXNfY29wcGVyIl0KCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDEwMjQqMTAyNCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9wYXRoKHJvb3QsIHJlbGF0aXZlKToKICAgIHBhdGggPSBQdXJlUG9zaXhQYXRoKHJlbGF0aXZlKQogICAgaWYgbm90IHJlbGF0aXZlIG9yIHBhdGguaXNfYWJzb2x1dGUoKSBvciAiLi4iIGluIHBhdGgucGFydHMgb3IgIlxcIiBpbiByZWxhdGl2ZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zYWZlIHBhdGg6IHtyZWxhdGl2ZX0iKQogICAgcmVzdWx0ID0gKFBhdGgocm9vdCkvcmVsYXRpdmUpLnJlc29sdmUoKQogICAgaWYgbm90IHJlc3VsdC5pc19yZWxhdGl2ZV90byhQYXRoKHJvb3QpLnJlc29sdmUoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBhdGggZXNjYXBlcyByb290OiB7cmVsYXRpdmV9IikKICAgIHJldHVybiByZXN1bHQKCgpkZWYgdmVyaWZ5X3ZpZXdzKHJvb3QsIG1hbmlmZXN0KToKICAgICIiIlZhbGlkYXRlIGZpbGVzLCBkZWNvZGVkIGdlb21ldHJ5LCBwYXJlbnQgc3BsaXRzLCBncm91cCBzcGxpdHMgYW5kIHRheG9ub215LiIiIgogICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICBpZiBtYW5pZmVzdFsibmFtZXMiXSAhPSBOQU1FUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDbGFzcyBvcmRlciBjaGFuZ2VkLiIpCiAgICBzZWVuLCBwYXJlbnRzLCBncm91cHMsIGhhc2hlcywgY2xhc3NlcyA9IHNldCgpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpLCBkZWZhdWx0ZGljdChzZXQpCiAgICBmb3Igcm93IGluIG1hbmlmZXN0WyJ2aWV3cyJdOgogICAgICAgIGRvbWFpbiwgc3BsaXQsIGtpbmQgPSByb3dbImRhdGFzZXQiXSwgcm93WyJzcGxpdCJdLCByb3dbImtpbmQiXQogICAgICAgIGlmIGRvbWFpbiBub3QgaW4gKCJzb3VyY2UiLCAidGFyZ2V0Iikgb3Igc3BsaXQgbm90IGluICgidHJhaW4iLCAidmFsIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVua25vd24gZG9tYWluIG9yIGRldmVsb3BtZW50LXRlc3QgZGF0YSBpbiBwYWNrYWdlLiIpCiAgICAgICAgaWYga2luZCBub3QgaW4gKCJmdWxsIiwgImNyb3AiLCAicmV2aWV3ZWRfYmFja2dyb3VuZCIpIG9yIChraW5kICE9ICJmdWxsIiBhbmQgKGRvbWFpbiAhPSAic291cmNlIiBvciBzcGxpdCAhPSAidHJhaW4iKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkRlcml2ZWQgdmlld3MgbXVzdCBiZSBzb3VyY2UgdHJhaW5pbmcgaW1hZ2VzLiIpCiAgICAgICAgaWYgbm90IHJvd1sicGFyZW50X2lkIl0uc3RhcnRzd2l0aChkb21haW4rIi8iKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUGFyZW50IGJlbG9uZ3MgdG8gYW5vdGhlciBkb21haW4uIikKICAgICAgICBpZiBraW5kID09ICJyZXZpZXdlZF9iYWNrZ3JvdW5kIiBhbmQgbm90IHJvdy5nZXQoInJldmlldyIpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJCYWNrZ3JvdW5kIGhhcyBubyByZXZpZXcgcmVjb3JkLiIpCiAgICAgICAgZm9yIGtleSBpbiAoImltYWdlIiwgImxhYmVsIik6CiAgICAgICAgICAgIGlmIHJvd1trZXldIGluIHNlZW46CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUgZmlsZSByZWZlcmVuY2UuIikKICAgICAgICAgICAgc2Vlbi5hZGQocm93W2tleV0pCiAgICAgICAgICAgIHBhdGggPSBzYWZlX3BhdGgocm9vdCwgcm93W2tleV0pCiAgICAgICAgICAgIGlmIGRpZ2VzdChwYXRoKSAhPSByb3dba2V5KyJfc2hhMjU2Il06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQ2hhbmdlZCB7a2V5fToge3Jvd1trZXldfScpCiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKHNhZmVfcGF0aChyb290LCByb3dbImltYWdlIl0pKSBhcyBpbToKICAgICAgICAgICAgaW0ubG9hZCgpCiAgICAgICAgICAgIGlmIGltLnNpemUgIT0gKHJvd1sid2lkdGgiXSwgcm93WyJoZWlnaHQiXSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJXcm9uZyBpbWFnZSBkaW1lbnNpb25zLiIpCiAgICAgICAgcGFyc2VkID0gW1tmbG9hdCh2KSBmb3IgdiBpbiBsaW5lLnNwbGl0KCldIGZvciBsaW5lIGluIHNhZmVfcGF0aChyb290LHJvd1sibGFiZWwiXSkucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgICAgICBpZiBsZW4ocGFyc2VkKSAhPSBsZW4ocm93WyJsYWJlbHMiXSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1hbmlmZXN0L2xhYmVsIGNvdW50IG1pc21hdGNoLiIpCiAgICAgICAgaWYgKG5vdCBwYXJzZWQpICE9IChraW5kID09ICJyZXZpZXdlZF9iYWNrZ3JvdW5kIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgZW1wdHkgb3Igbm9uZW1wdHkgYW5ub3RhdGlvbnMuIikKICAgICAgICBmb3IgdmFsdWVzLCBleHBlY3RlZCBpbiB6aXAocGFyc2VkLCByb3dbImxhYmVscyJdKToKICAgICAgICAgICAgaWYgbGVuKHZhbHVlcykgIT0gNSBvciBub3QgYWxsKG1hdGguaXNmaW5pdGUodikgZm9yIHYgaW4gdmFsdWVzKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgbGFiZWwgcm93LiIpCiAgICAgICAgICAgIGMsIHgsIHksIHcsIGggPSB2YWx1ZXMKICAgICAgICAgICAgYWxsb3dlZCA9IHJhbmdlKDYpIGlmIGRvbWFpbiA9PSAic291cmNlIiBlbHNlIHJhbmdlKDEsNikKICAgICAgICAgICAgaWYgYyAhPSBpbnQoYykgb3IgaW50KGMpIG5vdCBpbiBhbGxvd2VkIG9yIG1pbih3LGgpIDw9IDAgb3IgbWluKHgtdy8yLHktaC8yKSA8IC0xZS02IG9yIG1heCh4K3cvMix5K2gvMikgPiAxKzFlLTY6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIGNsYXNzIG9yIGJveCBnZW9tZXRyeS4iKQogICAgICAgICAgICBpZiBsZW4oZXhwZWN0ZWQpICE9IDUgb3Igbm90IGFsbChtYXRoLmlzZmluaXRlKHYpIGZvciB2IGluIGV4cGVjdGVkKSBvciBhbnkoYWJzKGEtYik+MWUtOCBmb3IgYSxiIGluIHppcCh2YWx1ZXMsZXhwZWN0ZWQpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1hbmlmZXN0L2xhYmVsIGdlb21ldHJ5IG1pc21hdGNoLiIpCiAgICAgICAgICAgIGNsYXNzZXNbKGRvbWFpbixzcGxpdCldLmFkZChpbnQoYykpCiAgICAgICAgcGFyZW50c1tyb3dbInBhcmVudF9pZCJdXS5hZGQoc3BsaXQpCiAgICAgICAgaWYgcm93WyJncm91cF9pZCJdIGlzIG5vdCBOb25lOgogICAgICAgICAgICBncm91cHNbKGRvbWFpbixyb3dbImdyb3VwX2lkIl0pXS5hZGQoc3BsaXQpCiAgICAgICAgaGFzaGVzW3Jvd1siaW1hZ2Vfc2hhMjU2Il1dLmFkZChzcGxpdCkKICAgIGlmIGFueShsZW4odik+MSBmb3IgcG9vbCBpbiAocGFyZW50cyxncm91cHMsaGFzaGVzKSBmb3IgdiBpbiBwb29sLnZhbHVlcygpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQYXJlbnQsIGdyb3VwIG9yIGV4YWN0LWltYWdlIGxlYWthZ2UgYWNyb3NzIHNwbGl0cy4iKQogICAgZm9yIGRvbWFpbiBpbiAoInNvdXJjZSIsICJ0YXJnZXQiKToKICAgICAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiKToKICAgICAgICAgICAgaWYgY2xhc3Nlc1soZG9tYWluLHNwbGl0KV0gIT0gc2V0KHJhbmdlKDYpIGlmIGRvbWFpbiA9PSAic291cmNlIiBlbHNlIHJhbmdlKDEsNikpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1pc3NpbmcgcmVxdWlyZWQgY2xhc3Nlczoge2RvbWFpbn0ve3NwbGl0fSIpCiAgICByZXR1cm4gZGljdChDb3VudGVyKGYne3JbImRhdGFzZXQiXX0ve3JbInNwbGl0Il19L3tyWyJraW5kIl19JyBmb3IgciBpbiBtYW5pZmVzdFsidmlld3MiXSkpCgoKZGVmIHZlcmlmeV9wYWNrYWdlKHJvb3QpOgogICAgcm9vdCA9IFBhdGgocm9vdCkKICAgIHJlY29yZCA9IGpzb24ubG9hZHMoKHJvb3QvInBhY2thZ2VfaGFzaGVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGZvciBuYW1lLCBjaGVja3N1bSBpbiByZWNvcmRbInNoYTI1NiJdLml0ZW1zKCk6CiAgICAgICAgaWYgZGlnZXN0KHNhZmVfcGF0aChyb290LG5hbWUpKSAhPSBjaGVja3N1bToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBhY2thZ2UgY2hlY2tzdW0gbWlzbWF0Y2g6IHtuYW1lfSIpCiAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMoKHJvb3QvIm1hbmlmZXN0Lmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHJlcXVpcmVkID0ge3Jba10gZm9yIHIgaW4gbWFuaWZlc3RbInZpZXdzIl0gZm9yIGsgaW4gKCJpbWFnZSIsImxhYmVsIil9CiAgICByZXF1aXJlZCB8PSB7Im1hbmlmZXN0Lmpzb24iLCAid2VpZ2h0cy95b2xvdjhuLnB0IiwgIndlaWdodHMveW9sb3Y4cy5wdCIsICJ3ZWlnaHRzL3NvdXJjZV9iYXNlbGluZS5wdCIsICJ3ZWlnaHRzL3RhcmdldF9iYXNlbGluZS5wdCJ9CiAgICBpZiBub3QgcmVxdWlyZWQgPD0gc2V0KHJlY29yZFsic2hhMjU2Il0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkluY29tcGxldGUgcGFja2FnZSBoYXNoIGNvdmVyYWdlLiIpCiAgICByZWZlcmVuY2VkID0ge3Jba10gZm9yIHIgaW4gbWFuaWZlc3RbInZpZXdzIl0gZm9yIGsgaW4gKCJpbWFnZSIsICJsYWJlbCIpfQogICAgYWN0dWFsID0ge3AucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSBmb3IgZm9sZGVyIGluICgiaW1hZ2VzIiwibGFiZWxzIikKICAgICAgICAgICAgICBmb3IgcCBpbiAocm9vdC9mb2xkZXIpLnJnbG9iKCIqIikgaWYgcC5pc19maWxlKCkgYW5kIHAuc3VmZml4ICE9ICIuY2FjaGUifQogICAgaWYgYWN0dWFsICE9IHJlZmVyZW5jZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5saXN0ZWQgaW1hZ2Ugb3IgbGFiZWwgaW4gcGFja2FnZS4iKQogICAgcmV0dXJuIHZlcmlmeV92aWV3cyhyb290LG1hbmlmZXN0KQoKCmRlZiBleHRyYWN0X3BhY2thZ2UoYXJjaGl2ZSwgZGVzdGluYXRpb24pOgogICAgIiIiRXh0cmFjdCBpbnRvIGEgbmV3IGZvbGRlciBvbmx5OyByZWplY3QgZHVwbGljYXRlLCB0cmF2ZXJzYWwgYW5kIHN5bWxpbmsgZW50cmllcy4iIiIKICAgIGRlc3RpbmF0aW9uID0gUGF0aChkZXN0aW5hdGlvbikucmVzb2x2ZSgpCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVc2UgYSBuZXcgZXh0cmFjdGlvbiBkaXJlY3Rvcnk7IGV4aXN0aW5nIGRhdGEgd2lsbCBub3QgYmUgb3ZlcndyaXR0ZW4uIikKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGFyY2hpdmUpIGFzIHo6CiAgICAgICAgbmFtZXMgPSB6Lm5hbWVsaXN0KCkKICAgICAgICBpZiBsZW4obmFtZXMpICE9IGxlbihzZXQobmFtZXMpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIFpJUCBlbnRyaWVzLiIpCiAgICAgICAgZm9yIGluZm8gaW4gei5pbmZvbGlzdCgpOgogICAgICAgICAgICBzYWZlX3BhdGgoZGVzdGluYXRpb24sIGluZm8uZmlsZW5hbWUpCiAgICAgICAgICAgIGlmIChpbmZvLmV4dGVybmFsX2F0dHIgPj4gMTYpICYgMG8xNzAwMDAgPT0gMG8xMjAwMDA6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTeW1saW5rIGluIFpJUC4iKQogICAgICAgIGlmIHoudGVzdHppcCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEYW1hZ2VkIFpJUC4iKQogICAgICAgIGRlc3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAgICB6LmV4dHJhY3RhbGwoZGVzdGluYXRpb24pCiAgICB2ZXJpZnlfcGFja2FnZShkZXN0aW5hdGlvbikKICAgIHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBjb25maWd1cmUocm9vdCwgb3V0cHV0KToKICAgICIiIlNlcGFyYXRlIHNvdXJjZS90YXJnZXQgcmVjaXBlczsgdmFsIGFsd2F5cyB1c2VzIHVuY2hhbmdlZCBvcmlnaW5hbCB2aWV3cy4iIiIKICAgIGltcG9ydCB5YW1sCiAgICByb290LCBvdXRwdXQgPSBQYXRoKHJvb3QpLnJlc29sdmUoKSwgUGF0aChvdXRwdXQpLnJlc29sdmUoKQogICAgdmVyaWZ5X3BhY2thZ2Uocm9vdCkKICAgIG1hbmlmZXN0ID0ganNvbi5sb2Fkcygocm9vdC8ibWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgY29uZmlncywgY29udGVudHMgPSB7fSwge30KICAgIGZvciByZWNpcGUsIGRvbWFpbiwgZnVsbF9vbmx5IGluIFsoInNvdXJjZV9mdWxsIiwic291cmNlIixUcnVlKSwgKCJzb3VyY2Vfdmlld3MiLCJzb3VyY2UiLEZhbHNlKSwgKCJ0YXJnZXQiLCJ0YXJnZXQiLFRydWUpXToKICAgICAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiKToKICAgICAgICAgICAgaW1hZ2VzID0gW3N0cihyb290L3JbImltYWdlIl0pIGZvciByIGluIG1hbmlmZXN0WyJ2aWV3cyJdIGlmIHJbImRhdGFzZXQiXT09ZG9tYWluIGFuZCByWyJzcGxpdCJdPT1zcGxpdCBhbmQgKG5vdCBmdWxsX29ubHkgb3Igclsia2luZCJdPT0iZnVsbCIpXQogICAgICAgICAgICBjb250ZW50c1tvdXRwdXQvZiJ7cmVjaXBlfV97c3BsaXR9LnR4dCJdID0gIlxuIi5qb2luKGltYWdlcykrIlxuIgogICAgICAgIHBhdGggPSBvdXRwdXQvZiJ7cmVjaXBlfS55YW1sIgogICAgICAgIGNvbnRlbnRzW3BhdGhdID0geWFtbC5zYWZlX2R1bXAoeyJwYXRoIjpzdHIocm9vdCksICJ0cmFpbiI6c3RyKG91dHB1dC9mIntyZWNpcGV9X3RyYWluLnR4dCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsIjpzdHIob3V0cHV0L2Yie3JlY2lwZX1fdmFsLnR4dCIpLCAibmFtZXMiOk5BTUVTfSkKICAgICAgICBjb25maWdzW3JlY2lwZV0gPSBzdHIocGF0aCkKICAgIGZvciBwYXRoLCBjb250ZW50IGluIGNvbnRlbnRzLml0ZW1zKCk6CiAgICAgICAgaWYgcGF0aC5leGlzdHMoKSBhbmQgcGF0aC5yZWFkX3RleHQoKSAhPSBjb250ZW50OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeGlzdGluZyBjb25maWd1cmF0aW9uIGRpZmZlcnM6ICIgKyBzdHIocGF0aCkpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBmb3IgcGF0aCwgY29udGVudCBpbiBjb250ZW50cy5pdGVtcygpOgogICAgICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICBwYXRoLndyaXRlX3RleHQoY29udGVudCkKICAgIHJldHVybiBjb25maWdzCg==', 'sha256': '1214793aa1513c82067430e9c5abcd48d8925b3175a29fa30fd3593bb4b16589'}, 'checkpoints.py': {'base64': 'IiIiRnJvemVuIGNoZWNrcG9pbnQgaGVscGVycyBhZGFwdGVkIGZyb20gYWNjdXJhY3kvdHJhaW5fcmVjb3ZlcnkucHkuCgpQZXItcnVuIGFyY2hpdmUgbmFtZXMgYXZvaWQgY29sbGlzaW9ucyBiZXR3ZWVuIHNwZWNpYWxpc3QgZXhwZXJpbWVudHMuClVzZSBvbmx5IHRydXN0ZWQsIGxvY2FsbHkgcHJvZHVjZWQgUHlUb3JjaCBjaGVja3BvaW50cy4KIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgaGFzaGxpYixqc29uLG9zLHppcGZpbGUsY3N2LGlvLHRlbXBmaWxlCgpjbGFzcyBUcmFpbmluZ1BhdXNlZChFeGNlcHRpb24pOgogIiIiTGVhdmUgdGhlIHRyYWluaW5nIGxvb3AgYmVmb3JlIGZpbmFsX2V2YWwgc3RyaXBzIHRoZSBvcHRpbWl6ZXIuIiIiCgoKZGVmIGNoZWNrcG9pbnRfcGF0aChydW4pOgogcmV0dXJuIFBhdGgocnVuKS53aXRoX25hbWUoUGF0aChydW4pLm5hbWUgKyAnX2NoZWNrcG9pbnQuemlwJykKCgpkZWYgcmVzdG9yZV9jaGVja3BvaW50KGFyY2hpdmUscnVuLHNpZ25hdHVyZT1Ob25lKToKICIiIlZlcmlmeSB0aGUgZW50aXJlIGJhY2t1cCBiZWZvcmUgcmVwbGFjaW5nIGFueSBydW4gZmlsZXMuIE93biBjaGVja3BvaW50cyBvbmx5LiIiIgogYXJjaGl2ZT1QYXRoKGFyY2hpdmUpO3J1bj1QYXRoKHJ1bikucmVzb2x2ZSgpCiB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmNoaXZlKSBhcyB6OgogIG5hbWVzPXoubmFtZWxpc3QoKQogIGlmIGxlbihuYW1lcykhPWxlbihzZXQobmFtZXMpKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0R1cGxpY2F0ZSBjaGVja3BvaW50IGFyY2hpdmUgZW50cmllcy4nKQogIGZvciBuYW1lIGluIG5hbWVzOgogICBpZiBQYXRoKG5hbWUpLmlzX2Fic29sdXRlKCkgb3IgJ1xcJyBpbiBuYW1lIG9yICcuLicgaW4gUGF0aChuYW1lKS5wYXJ0cyBvciBub3QgKHJ1bi9uYW1lKS5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocnVuKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1Vuc2FmZSBjaGVja3BvaW50IHBhdGguJykKICBmb3IgaW5mbyBpbiB6LmluZm9saXN0KCk6CiAgIGlmICgoaW5mby5leHRlcm5hbF9hdHRyID4+IDE2KSAmIDBvMTcwMDAwKSA9PSAwbzEyMDAwMDpyYWlzZSBSdW50aW1lRXJyb3IoJ1N5bWxpbmsgaW4gY2hlY2twb2ludC4nKQogIGlmIHoudGVzdHppcCgpIGlzIG5vdCBOb25lOnJhaXNlIFJ1bnRpbWVFcnJvcignQ2hlY2twb2ludCBaSVAgaXMgZGFtYWdlZC4nKQogIHJlY29yZD1qc29uLmxvYWRzKHoucmVhZCgnc2lnbmF0dXJlLmpzb24nKSkKICBleHBlY3RlZD1zaWduYXR1cmUKICBpZiAocnVuLydzaWduYXR1cmUuanNvbicpLmV4aXN0cygpOgogICBleGlzdGluZz1qc29uLmxvYWRzKChydW4vJ3NpZ25hdHVyZS5qc29uJykucmVhZF90ZXh0KCkpWydzaGEyNTYnXQogICBpZiBleHBlY3RlZCBpcyBub3QgTm9uZSBhbmQgZXhpc3RpbmchPWV4cGVjdGVkOnJhaXNlIFJ1bnRpbWVFcnJvcignUnVuIHNpZ25hdHVyZSBtaXNtYXRjaC4nKQogICBleHBlY3RlZD1leGlzdGluZwogIGlmIGV4cGVjdGVkIGlzIG5vdCBOb25lIGFuZCByZWNvcmRbJ3NoYTI1NiddIT1leHBlY3RlZDpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgZGF0YXNldC9yZWNpcGUgbWlzbWF0Y2guJykKICBmb3IgcmVxdWlyZWQgaW4gWyd0cmFpbmluZy93ZWlnaHRzL2xhc3QucHQnLCd0cmFpbmluZy93ZWlnaHRzL2Jlc3QucHQnLCd0cmFpbmluZy9yZXN1bHRzLmNzdicsJ3RyYWluaW5nL2FyZ3MueWFtbCcsJ3RyYWluaW5nL3RyYWluZXJfc3RhdGUucHQnLCdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nXToKICAgaWYgcmVxdWlyZWQgbm90IGluIG5hbWVzOnJhaXNlIFJ1bnRpbWVFcnJvcignSW5jb21wbGV0ZSBjaGVja3BvaW50OiAnK3JlcXVpcmVkKQogIGlmICdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nIGluIG5hbWVzOgogICBtYW5pZmVzdD1qc29uLmxvYWRzKHoucmVhZCgnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJykpCiAgIGFjdHVhbD1zZXQobmFtZXMpLXsnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJ30KICAgaWYgYWN0dWFsIT1zZXQobWFuaWZlc3RbJ3NoYTI1NiddKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgZmlsZSBsaXN0IG1pc21hdGNoLicpCiAgIGZvciBuYW1lLGRpZ2VzdCBpbiBtYW5pZmVzdFsnc2hhMjU2J10uaXRlbXMoKToKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHoucmVhZChuYW1lKSkuaGV4ZGlnZXN0KCkhPWRpZ2VzdDpyYWlzZSBSdW50aW1lRXJyb3IoJ0NoZWNrcG9pbnQgY2hlY2tzdW0gbWlzbWF0Y2g6ICcrbmFtZSkKICBydW4ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KGRpcj1ydW4ucGFyZW50LHByZWZpeD0nY2hlY2twb2ludC1yZXN0b3JlLScpIGFzIHRlbXBvcmFyeToKICAgc3RhZ2luZz1QYXRoKHRlbXBvcmFyeSkKICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICBpZiBuYW1lLmVuZHN3aXRoKCcvJyk6Y29udGludWUKICAgIHRhcmdldD1zdGFnaW5nL25hbWU7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7dGFyZ2V0LndyaXRlX2J5dGVzKHoucmVhZChuYW1lKSkKICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICBpZiBuYW1lLmVuZHN3aXRoKCcvJyk6Y29udGludWUKICAgIHRhcmdldD1ydW4vbmFtZTt0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKTtvcy5yZXBsYWNlKHN0YWdpbmcvbmFtZSx0YXJnZXQpCiBwcmludCgnVmVyaWZpZWQgY2hlY2twb2ludCByZXN0b3JlZDonLGFyY2hpdmUsZmx1c2g9VHJ1ZSkKCgpkZWYgc2F2ZV9jaGVja3BvaW50KHRyYWluZXIscnVuKToKICIiIlB1Ymxpc2ggYSBjb21wbGV0ZSwgY2hlY2tzdW1tZWQgYmFja3VwIGF0b21pY2FsbHkgYWZ0ZXIgZXZlcnkgc2F2ZWQgZXBvY2guIiIiCiBpbXBvcnQgdG9yY2gscmFuZG9tLG51bXB5IGFzIG5wCiBydW49UGF0aChydW4pO2Vwb2NoPWludCh0cmFpbmVyLmVwb2NoKQogc3RhdGU9eydlcG9jaCc6ZXBvY2gsJ3RyYWluaW5nX2ZpbmlzaGVkJzpib29sKHRyYWluZXIuc3RvcCksCiAgJ3N0b3BwZXInOntrZXk6Z2V0YXR0cih0cmFpbmVyLnN0b3BwZXIsa2V5KSBmb3Iga2V5IGluIFsnYmVzdF9maXRuZXNzJywnYmVzdF9lcG9jaCcsJ3Bvc3NpYmxlX3N0b3AnLCdwYXRpZW5jZSddfSwKICAnc2NoZWR1bGVyJzp0cmFpbmVyLnNjaGVkdWxlci5zdGF0ZV9kaWN0KCksJ3B5dGhvbl9ybmcnOnJhbmRvbS5nZXRzdGF0ZSgpLAogICdudW1weV9ybmcnOm5wLnJhbmRvbS5nZXRfc3RhdGUoKSwndG9yY2hfcm5nJzp0b3JjaC5nZXRfcm5nX3N0YXRlKCksCiAgJ2N1ZGFfcm5nJzp0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGVfYWxsKCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmV9CiBzdGF0ZV9maWxlPXJ1bi8ndHJhaW5pbmcvdHJhaW5lcl9zdGF0ZS5wdCc7c3RhdGVfZmlsZS5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiBzdGF0ZV90bXA9c3RhdGVfZmlsZS53aXRoX3N1ZmZpeCgnLnRtcCcpO3RvcmNoLnNhdmUoc3RhdGUsc3RhdGVfdG1wKTtvcy5yZXBsYWNlKHN0YXRlX3RtcCxzdGF0ZV9maWxlKQogZmlsZXM9W1BhdGgodHJhaW5lci5sYXN0KSxQYXRoKHRyYWluZXIuYmVzdCkscnVuLyd0cmFpbmluZy9hcmdzLnlhbWwnLHJ1bi8ndHJhaW5pbmcvcmVzdWx0cy5jc3YnLHJ1bi8nc2lnbmF0dXJlLmpzb24nLHN0YXRlX2ZpbGVdCiBmaWxlcy5leHRlbmQocCBmb3IgcCBpbiBbcnVuLydiYXNlbGluZV9zb3VyY2VfdmFsaWRhdGlvbi5qc29uJyxydW4vJ2Jhc2VsaW5lX3RhcmdldF92YWxpZGF0aW9uLmpzb24nXSBpZiBwLmlzX2ZpbGUoKSkKIGlmIGFueShub3QgcC5pc19maWxlKCkgZm9yIHAgaW4gZmlsZXMpOnJhaXNlIFJ1bnRpbWVFcnJvcignQ2Fubm90IHB1Ymxpc2ggYW4gaW5jb21wbGV0ZSBjaGVja3BvaW50LicpCiBhcmNoaXZlPWNoZWNrcG9pbnRfcGF0aChydW4pO3RlbXBvcmFyeT1hcmNoaXZlLndpdGhfc3VmZml4KCcudG1wJykKIHRyeToKICBoYXNoZXM9e30KICB3aXRoIHppcGZpbGUuWmlwRmlsZSh0ZW1wb3JhcnksJ3cnLHppcGZpbGUuWklQX0RFRkxBVEVELGNvbXByZXNzbGV2ZWw9MSkgYXMgejoKICAgZm9yIHAgaW4gZmlsZXM6CiAgICBwYXlsb2FkPXAucmVhZF9ieXRlcygpO25hbWU9cC5yZWxhdGl2ZV90byhydW4pLmFzX3Bvc2l4KCk7aGFzaGVzW25hbWVdPWhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpO3oud3JpdGVzdHIobmFtZSxwYXlsb2FkKQogICB6LndyaXRlc3RyKCdjaGVja3BvaW50X21hbmlmZXN0Lmpzb24nLGpzb24uZHVtcHMoeydmb3JtYXQnOidwY2JfcGF1c2VfcmVzdW1lX3YxJywnY29tcGxldGVkX2Vwb2Nocyc6ZXBvY2grMSwndHJhaW5pbmdfZmluaXNoZWQnOnN0YXRlWyd0cmFpbmluZ19maW5pc2hlZCddLCdzaGEyNTYnOmhhc2hlc30pKQogIHdpdGggemlwZmlsZS5aaXBGaWxlKHRlbXBvcmFyeSkgYXMgejoKICAgaWYgei50ZXN0emlwKCkgaXMgbm90IE5vbmU6cmFpc2UgUnVudGltZUVycm9yKCdDaGVja3BvaW50IHZlcmlmaWNhdGlvbiBmYWlsZWQuJykKICBvcy5yZXBsYWNlKHRlbXBvcmFyeSxhcmNoaXZlKQogZmluYWxseTp0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKIHByaW50KGYnQ0hFQ0tQT0lOVCBTQVZFRDogZXBvY2gge2Vwb2NoKzF9IOKAlCB7YXJjaGl2ZX0nLGZsdXNoPVRydWUpCiByZXR1cm4gYXJjaGl2ZQoKCmRlZiByZXN0b3JlX3RyYWluZXJfc3RhdGUodHJhaW5lcixydW4pOgogaW1wb3J0IHRvcmNoLHJhbmRvbSxudW1weSBhcyBucAogcGF0aD1QYXRoKHJ1bikvJ3RyYWluaW5nL3RyYWluZXJfc3RhdGUucHQnCiBpZiBub3QgcGF0aC5leGlzdHMoKSBvciBub3QgdHJhaW5lci5yZXN1bWU6cmV0dXJuCiBzdGF0ZT10b3JjaC5sb2FkKHBhdGgsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSkKIGlmIHN0YXRlWydlcG9jaCddKzEhPXRyYWluZXIuc3RhcnRfZXBvY2g6cmFpc2UgUnVudGltZUVycm9yKCdDaGVja3BvaW50IGFuZCB0cmFpbmVyIHN0YXRlIHJlZmVyIHRvIGRpZmZlcmVudCBlcG9jaHMuJykKIGZvciBrZXksdmFsdWUgaW4gc3RhdGVbJ3N0b3BwZXInXS5pdGVtcygpOnNldGF0dHIodHJhaW5lci5zdG9wcGVyLGtleSx2YWx1ZSkKIHRyYWluZXIuc2NoZWR1bGVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnc2NoZWR1bGVyJ10pCiByYW5kb20uc2V0c3RhdGUoc3RhdGVbJ3B5dGhvbl9ybmcnXSk7bnAucmFuZG9tLnNldF9zdGF0ZShzdGF0ZVsnbnVtcHlfcm5nJ10pO3RvcmNoLnNldF9ybmdfc3RhdGUoc3RhdGVbJ3RvcmNoX3JuZyddKQogaWYgc3RhdGVbJ2N1ZGFfcm5nJ10gaXMgbm90IE5vbmUgYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6dG9yY2guY3VkYS5zZXRfcm5nX3N0YXRlX2FsbChzdGF0ZVsnY3VkYV9ybmcnXSkKCgpkZWYgdHJpbV9yZXN1bHRzKHJ1bixjb21wbGV0ZWRfZXBvY2hzKToKICIiIkRpc2NhcmQgQ1NWIHJvd3Mgd3JpdHRlbiBieSBhbiBlcG9jaCB3aG9zZSB3ZWlnaHRzIHdlcmUgbm90IGNvbW1pdHRlZC4iIiIKIHBhdGg9UGF0aChydW4pLyd0cmFpbmluZy9yZXN1bHRzLmNzdicKIGlmIG5vdCBwYXRoLmV4aXN0cygpOnJldHVybgogd2l0aCBwYXRoLm9wZW4obmV3bGluZT0nJykgYXMgc3RyZWFtOgogIHJlYWRlcj1jc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKTtmaWVsZHM9cmVhZGVyLmZpZWxkbmFtZXMKICByb3dzPVtyb3cgZm9yIHJvdyBpbiByZWFkZXIgaWYgaW50KGZsb2F0KHJvd1snZXBvY2gnXSkpPD1jb21wbGV0ZWRfZXBvY2hzXQogb3V0cHV0PWlvLlN0cmluZ0lPKCk7d3JpdGVyPWNzdi5EaWN0V3JpdGVyKG91dHB1dCxmaWVsZG5hbWVzPWZpZWxkcyk7d3JpdGVyLndyaXRlaGVhZGVyKCk7d3JpdGVyLndyaXRlcm93cyhyb3dzKQogcGF0aC53cml0ZV90ZXh0KG91dHB1dC5nZXR2YWx1ZSgpKQoK', 'sha256': 'd235ebf57d1d4581e9cc38d52ea6fe727ff333f0b620ffb73286769cc9dbc419'}, 'smoke.py': {'base64': 'IiIiVGlueSB0cmFpbmluZy1zZXQgbWVtb3JpemF0aW9uIGFuZCBmcmVzaC1wcm9jZXNzIHJlc3VtZSBjaGVjaywgTk9UIGV2YWx1YXRpb24uCgpVc2VzIHNpeCB0cmFpbmluZyBjcm9wcywgb25lIHBlciBjbGFzcy4gTm8gdmFsaWRhdGlvbiBvciB0ZXN0IGltYWdlcyBhcmUgdXNlZC4KVGhlIGRlbGliZXJhdGVseSByZXBlYXRlZCB0cmFpbi9ldmFsIHNhbXBsZSB0ZXN0cyB3aGV0aGVyIHRoZSBwaXBlbGluZSBjYW4gbGVhcm4uCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKZnJvbSByZXZhbXAucnVudGltZSBpbXBvcnQgZGlnZXN0LCB2ZXJpZnlfcGFja2FnZQoKUkVDSVBFID0gZGljdChlcG9jaHM9ODAsIHBhdGllbmNlPTEwMCwgaW1nc3o9MzIwLCBiYXRjaD02LCB3b3JrZXJzPTAsCiAgICAgICAgICAgICAgb3B0aW1pemVyPSJBZGFtVyIsIGxyMD0uMDAzLCBscmY9LjEsIG5icz02LCBjb3NfbHI9VHJ1ZSwKICAgICAgICAgICAgICB3YXJtdXBfZXBvY2hzPTAsIHNlZWQ9NDIsIGRldGVybWluaXN0aWM9VHJ1ZSwgYW1wPUZhbHNlLAogICAgICAgICAgICAgIG1vc2FpYz0wLCBtaXh1cD0wLCBjb3B5X3Bhc3RlPTAsIGRlZ3JlZXM9MCwgdHJhbnNsYXRlPTAsIHNjYWxlPTAsCiAgICAgICAgICAgICAgZmxpcGxyPTAsIGZsaXB1ZD0wLCBoc3ZfaD0wLCBoc3Zfcz0wLCBoc3Zfdj0wLAogICAgICAgICAgICAgIGNsb3NlX21vc2FpYz0wLCBzYXZlPVRydWUsIHNhdmVfcGVyaW9kPS0xLCBwbG90cz1GYWxzZSkKCgpkZWYgc2lnbmF0dXJlKHBhY2thZ2UpOgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdWx0cmFseXRpY3MKICAgIHJldHVybiBoYXNobGliLnNoYTI1NigoZGlnZXN0KFBhdGgocGFja2FnZSkvInBhY2thZ2VfaGFzaGVzLmpzb24iKSArIGpzb24uZHVtcHMoUkVDSVBFLCBzb3J0X2tleXM9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgKyB0b3JjaC5fX3ZlcnNpb25fXyArIHVsdHJhbHl0aWNzLl9fdmVyc2lvbl9fKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgd29ya2VyKHBhY2thZ2UsIGRhdGEsIHJ1biwgcmVzdW1lLCBkZXZpY2UpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBmcm9tIHJldmFtcCBpbXBvcnQgY2hlY2twb2ludHMgYXMgY2sKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcyg0KQogICAgcnVuID0gUGF0aChydW4pCiAgICBleHBlY3RlZCA9IHNpZ25hdHVyZShwYWNrYWdlKQogICAgaWYgcmVzdW1lOgogICAgICAgIGlmIGpzb24ubG9hZHMoKHJ1bi8ic2lnbmF0dXJlLmpzb24iKS5yZWFkX3RleHQoKSlbInNoYTI1NiJdICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGVja3BvaW50IHNpZ25hdHVyZSBkaWZmZXJzIGZyb20gZGF0YXNldC9yZWNpcGUvZW52aXJvbm1lbnQuIikKICAgICAgICByYXcgPSB0b3JjaC5sb2FkKHJ1bi8idHJhaW5pbmcvd2VpZ2h0cy9sYXN0LnB0IiwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgY29tcGxldGVkID0gaW50KHJhd1siZXBvY2giXSkgKyAxCiAgICAgICAgaWYgY29tcGxldGVkIDwgMSBvciByYXdbIm9wdGltaXplciJdIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGEgY29tcGxldGVkIGVwb2NoIHdpdGggcmVzdW1hYmxlIG9wdGltaXplciBzdGF0ZS4iKQogICAgICAgIGRlbCByYXcKICAgICAgICBjay50cmltX3Jlc3VsdHMocnVuLCBjb21wbGV0ZWQpCiAgICAgICAgaWYgY29tcGxldGVkID49IFJFQ0lQRVsiZXBvY2hzIl06CiAgICAgICAgICAgIHByaW50KCJBbGwgZXBvY2hzIGFscmVhZHkgc2F2ZWQ7IHJlYnVpbGRpbmcgdGhlIHJlcG9ydCB3aXRob3V0IHJldHJhaW5pbmcuIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmV0dXJuCiAgICBlbHNlOgogICAgICAgIHJ1bi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgICAgIChydW4vInNpZ25hdHVyZS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHsic2hhMjU2IjpleHBlY3RlZCwgInJlY2lwZSI6UkVDSVBFfSkpCiAgICBtb2RlbCA9IFlPTE8oc3RyKHJ1bi8idHJhaW5pbmcvd2VpZ2h0cy9sYXN0LnB0IiBpZiByZXN1bWUgZWxzZSBQYXRoKHBhY2thZ2UpLyJ3ZWlnaHRzL3lvbG92OG4ucHQiKSkKICAgIGRlZiBzYXZlKHRyYWluZXIpOgogICAgICAgIGNrLnNhdmVfY2hlY2twb2ludCh0cmFpbmVyLCBydW4pCiAgICBkZWYgcGF1c2UodHJhaW5lcik6CiAgICAgICAgaWYgbm90IHJlc3VtZSBhbmQgdHJhaW5lci5lcG9jaCA9PSAwOgogICAgICAgICAgICByYWlzZSBjay5UcmFpbmluZ1BhdXNlZCgpCiAgICBkZWYgcmVzdG9yZWQodHJhaW5lcik6CiAgICAgICAgY2sucmVzdG9yZV90cmFpbmVyX3N0YXRlKHRyYWluZXIscnVuKQogICAgICAgIGlmIHJlc3VtZToKICAgICAgICAgICAgaWYgdHJhaW5lci5zdGFydF9lcG9jaCAhPSBjb21wbGV0ZWQgb3Igbm90IHRyYWluZXIub3B0aW1pemVyLnN0YXRlOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJPcHRpbWl6ZXIvZXBvY2ggd2FzIG5vdCByZXN0b3JlZC4iKQogICAgICAgICAgICAocnVuLyJyZXN1bWVfdmVyaWZpZWQuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7InN0YXJ0X2Vwb2NoIjp0cmFpbmVyLnN0YXJ0X2Vwb2NoLAogICAgICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZV9lbnRyaWVzIjpsZW4odHJhaW5lci5vcHRpbWl6ZXIuc3RhdGUpLCAicGlkIjpvcy5nZXRwaWQoKX0pKQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCJvbl9tb2RlbF9zYXZlIixzYXZlKQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCJvbl9maXRfZXBvY2hfZW5kIixwYXVzZSkKICAgIG1vZGVsLmFkZF9jYWxsYmFjaygib25fdHJhaW5fc3RhcnQiLHJlc3RvcmVkKQogICAgdHJ5OgogICAgICAgIGlmIHJlc3VtZToKICAgICAgICAgICAgbW9kZWwudHJhaW4ocmVzdW1lPVRydWUsIGRldmljZT1kZXZpY2UsIGRhdGE9c3RyKGRhdGEpLCBzYXZlX2Rpcj1zdHIocnVuLyJ0cmFpbmluZyIpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG1vZGVsLnRyYWluKGRhdGE9c3RyKGRhdGEpLCBkZXZpY2U9ZGV2aWNlLCBwcm9qZWN0PXN0cihydW4pLCBuYW1lPSJ0cmFpbmluZyIsIGV4aXN0X29rPVRydWUsICoqUkVDSVBFKQogICAgZXhjZXB0IGNrLlRyYWluaW5nUGF1c2VkOgogICAgICAgIHJldHVybgogICAgZmluYWxseToKICAgICAgICB0cmFpbmVyID0gZ2V0YXR0cihtb2RlbCwidHJhaW5lciIsTm9uZSkKICAgICAgICBmb3IgbG9hZGVyIGluIFtnZXRhdHRyKHRyYWluZXIsInRyYWluX2xvYWRlciIsTm9uZSksZ2V0YXR0cih0cmFpbmVyLCJ0ZXN0X2xvYWRlciIsTm9uZSldOgogICAgICAgICAgICBpZiBjYWxsYWJsZShnZXRhdHRyKGxvYWRlciwiY2xvc2UiLE5vbmUpKToKICAgICAgICAgICAgICAgIGxvYWRlci5jbG9zZSgpCgoKZGVmIGNoZWNrKHBhY2thZ2UsIG91dHB1dCwgZGV2aWNlPSJjcHUiLCBjaGVja3BvaW50PU5vbmUsIHJlc3VtZV9vbmx5PUZhbHNlKToKICAgICIiIlJlc3RhcnQtc2FmZSBjb29yZGluYXRvci4gVGhlIGxhc3QgYXRvbWljYWxseSBwdWJsaXNoZWQgWklQIGlzIGF1dGhvcml0YXRpdmUuIiIiCiAgICBpbXBvcnQgZmNudGwKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0KS5yZXNvbHZlKCkKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIChvdXRwdXQvInJ1bi5sb2NrIikub3BlbigiYSIpIGFzIGxvY2s6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmY250bC5mbG9jayhsb2NrLCBmY250bC5MT0NLX0VYIHwgZmNudGwuTE9DS19OQikKICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoaXMgcnVuIGlzIGFscmVhZHkgYWN0aXZlOyBkbyBub3QgbGF1bmNoIGEgc2Vjb25kIHdvcmtlci4iKSBmcm9tIGVycm9yCiAgICAgICAgcmV0dXJuIF9jaGVjayhwYWNrYWdlLCBvdXRwdXQsIGRldmljZSwgY2hlY2twb2ludCwgcmVzdW1lX29ubHkpCgoKZGVmIGNoZWNrcG9pbnRfYXJjaGl2ZShwYXRoLCBvdXRwdXQpOgogICAgIiIiQWNjZXB0IGEgZG93bmxvYWRlZCBaSVAgb3IgYSBLYWdnbGUtZXhwYW5kZWQgY2hlY2twb2ludCBkaXJlY3RvcnkuIiIiCiAgICBpbXBvcnQgemlwZmlsZQogICAgZnJvbSByZXZhbXAucnVudGltZSBpbXBvcnQgc2FmZV9wYXRoCiAgICBwYXRoID0gUGF0aChwYXRoKS5yZXNvbHZlKCkKICAgIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLnN1ZmZpeCA9PSAiLnppcCI6CiAgICAgICAgcmV0dXJuIHBhdGgKICAgIGlmIHBhdGgubmFtZSA9PSAiY2hlY2twb2ludF9tYW5pZmVzdC5qc29uIjoKICAgICAgICBwYXRoID0gcGF0aC5wYXJlbnQKICAgIHJlY29yZCA9IGpzb24ubG9hZHMoKHBhdGgvImNoZWNrcG9pbnRfbWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgZm9yIG5hbWUsIGV4cGVjdGVkIGluIHJlY29yZFsic2hhMjU2Il0uaXRlbXMoKToKICAgICAgICBpZiBkaWdlc3Qoc2FmZV9wYXRoKHBhdGgsbmFtZSkpICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGFuZ2VkIGV4cGFuZGVkIGNoZWNrcG9pbnQ6ICIgKyBuYW1lKQogICAgdGFyZ2V0ID0gUGF0aChvdXRwdXQpLyJpbXBvcnRlZF9jaGVja3BvaW50LnppcCIKICAgIHRlbXBvcmFyeSA9IHRhcmdldC53aXRoX3N1ZmZpeCgiLnRtcCIpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh0ZW1wb3JhcnksInciLHppcGZpbGUuWklQX0RFRkxBVEVELGNvbXByZXNzbGV2ZWw9MSkgYXMgejoKICAgICAgICBmb3IgbmFtZSBpbiBbKnJlY29yZFsic2hhMjU2Il0sImNoZWNrcG9pbnRfbWFuaWZlc3QuanNvbiJdOgogICAgICAgICAgICB6LndyaXRlKHNhZmVfcGF0aChwYXRoLG5hbWUpLG5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZSh0YXJnZXQpCiAgICByZXR1cm4gdGFyZ2V0CgoKZGVmIF9jaGVjayhwYWNrYWdlLCBvdXRwdXQsIGRldmljZSwgY2hlY2twb2ludCwgcmVzdW1lX29ubHkpOgogICAgaW1wb3J0IHlhbWwKICAgIGZyb20gcmV2YW1wIGltcG9ydCBjaGVja3BvaW50cyBhcyBjawogICAgcGFja2FnZSwgb3V0cHV0ID0gUGF0aChwYWNrYWdlKS5yZXNvbHZlKCksIFBhdGgob3V0cHV0KS5yZXNvbHZlKCkKICAgIHZlcmlmeV9wYWNrYWdlKHBhY2thZ2UpCiAgICBleHBlY3RlZCA9IHNpZ25hdHVyZShwYWNrYWdlKQogICAgbWFya2VyID0gb3V0cHV0LyJydW5fc2lnbmF0dXJlLmpzb24iCiAgICBpZiBtYXJrZXIuZXhpc3RzKCkgYW5kIGpzb24ubG9hZHMobWFya2VyLnJlYWRfdGV4dCgpKVsic2hhMjU2Il0gIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhpc3RpbmcgcnVuIHVzZXMgYW5vdGhlciBwYWNrYWdlLCByZWNpcGUgb3IgZW52aXJvbm1lbnQuIikKICAgIGlmIChvdXRwdXQvInJlcG9ydC5qc29uIikuZXhpc3RzKCk6CiAgICAgICAgaWYgbm90IG1hcmtlci5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29tcGxldGVkIHJ1biBsYWNrcyBhIG1hdGNoaW5nIHJlc3VtZSBtYXJrZXI7IHVzZSBhIG5ldyBvdXRwdXQgZGlyZWN0b3J5LiIpCiAgICAgICAgcmVwb3J0ID0ganNvbi5sb2Fkcygob3V0cHV0LyJyZXBvcnQuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHJlcG9ydFsicGFja2FnZV9oYXNoZXNfc2hhMjU2Il0gIT0gZGlnZXN0KHBhY2thZ2UvInBhY2thZ2VfaGFzaGVzLmpzb24iKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVwb3J0L3BhY2thZ2UgbWlzbWF0Y2guIikKICAgICAgICBpZiByZXBvcnRbInN0YXR1cyJdICE9ICJwYXNzZWQiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvbXBsZXRlZCBsZWFybmluZyBjaGVjayBmYWlsZWQ7IGluc3BlY3QgaXRzIHJlcG9ydCByYXRoZXIgdGhhbiBzaWxlbnRseSByZXRyYWluaW5nLiIpCiAgICAgICAgcHJpbnQoIkFscmVhZHkgY29tcGxldGU7IHJldXNpbmcgcmVwb3J0IHdpdGhvdXQgdHJhaW5pbmcuIiwgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gcmVwb3J0CiAgICBmaXJzdCwgZnJlc2ggPSBvdXRwdXQvImZpcnN0X3J1biIsIG91dHB1dC8iZnJlc2hfcnVuIgogICAgbG9jYWwgPSBjay5jaGVja3BvaW50X3BhdGgoZnJlc2gpIGlmIGNrLmNoZWNrcG9pbnRfcGF0aChmcmVzaCkuZXhpc3RzKCkgZWxzZSBjay5jaGVja3BvaW50X3BhdGgoZmlyc3QpCiAgICBpZiBjaGVja3BvaW50IGFuZCBsb2NhbC5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJMb2NhbCBwcm9ncmVzcyBleGlzdHMuIENsZWFyIFJFU1RPUkVfRlJPTSB0byByZXN1bWUgaXQ7IHVzZSBhIG5ldyBydW4gZm9sZGVyIGZvciBhbiBpbXBvcnRlZCBjaGVja3BvaW50LiIpCiAgICBpZiByZXN1bWVfb25seSBhbmQgbm90IGNoZWNrcG9pbnQgYW5kIG5vdCBsb2NhbC5leGlzdHMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZSByZXF1ZXN0ZWQgYnV0IG5vIGNoZWNrcG9pbnQgZXhpc3RzLiBObyB0cmFpbmluZyB3YXMgc3RhcnRlZC4iKQogICAgaWYgY2hlY2twb2ludDoKICAgICAgICBpbXBvcnRlZCA9IGNoZWNrcG9pbnRfYXJjaGl2ZShjaGVja3BvaW50LCBvdXRwdXQpCiAgICAgICAgY2sucmVzdG9yZV9jaGVja3BvaW50KGltcG9ydGVkLCBmcmVzaCwgZXhwZWN0ZWQpCiAgICAgICAgIyBQdWJsaXNoIHRoZSB2ZXJpZmllZCBpbXBvcnRlZCBhcmNoaXZlIGxvY2FsbHkgYmVmb3JlIGxhdW5jaGluZyBhbnkgd29yay4KICAgICAgICBzaHV0aWwuY29weWZpbGUoaW1wb3J0ZWQsIGNrLmNoZWNrcG9pbnRfcGF0aChmcmVzaCkpCiAgICAgICAgbG9jYWwgPSBjay5jaGVja3BvaW50X3BhdGgoZnJlc2gpCiAgICBtYXJrZXIud3JpdGVfdGV4dChqc29uLmR1bXBzKHsic2hhMjU2IjpleHBlY3RlZCwicmVjaXBlIjpSRUNJUEV9KSkKICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcygocGFja2FnZS8ibWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgc2VsZWN0ZWQgPSB7fQogICAgZm9yIHJvdyBpbiBtYW5pZmVzdFsidmlld3MiXToKICAgICAgICBpZiByb3dbImRhdGFzZXQiXT09InNvdXJjZSIgYW5kIHJvd1sic3BsaXQiXT09InRyYWluIiBhbmQgcm93WyJraW5kIl09PSJjcm9wIjoKICAgICAgICAgICAgY2xhc3NlcyA9IHtpbnQoYm94WzBdKSBmb3IgYm94IGluIHJvd1sibGFiZWxzIl19CiAgICAgICAgICAgIGlmIGxlbihjbGFzc2VzKT09MToKICAgICAgICAgICAgICAgIHNlbGVjdGVkLnNldGRlZmF1bHQobmV4dChpdGVyKGNsYXNzZXMpKSxyb3cpCiAgICBpZiBzZXQoc2VsZWN0ZWQpICE9IHNldChyYW5nZSg2KSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2Fubm90IGZvcm0gYSBzaXgtY2xhc3MgbGVhcm5pbmcgY2hlY2sgZnJvbSB0cmFpbmluZyBjcm9wcy4iKQogICAgZGF0YSA9IG91dHB1dC8idGlueSI7IChkYXRhLyJpbWFnZXMiKS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7IChkYXRhLyJsYWJlbHMiKS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZm9yIGNscywgcm93IGluIHNvcnRlZChzZWxlY3RlZC5pdGVtcygpKToKICAgICAgICBmb3Iga2V5LCBkZXN0aW5hdGlvbiBpbiBbKCJpbWFnZSIsZGF0YS9mImltYWdlcy97Y2xzfS5qcGciKSwoImxhYmVsIixkYXRhL2YibGFiZWxzL3tjbHN9LnR4dCIpXToKICAgICAgICAgICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBpZiBkaWdlc3QoZGVzdGluYXRpb24pICE9IHJvd1trZXkrIl9zaGEyNTYiXToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaGFuZ2VkIHRpbnkgbGVhcm5pbmcgaW5wdXQ6ICIgKyBzdHIoZGVzdGluYXRpb24pKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlKHBhY2thZ2Uvcm93W2tleV0sZGVzdGluYXRpb24pCiAgICBjZmcgPSBkYXRhLyJkYXRhLnlhbWwiCiAgICBjZmcud3JpdGVfdGV4dCh5YW1sLnNhZmVfZHVtcCh7InBhdGgiOnN0cihkYXRhKSwgInRyYWluIjoiaW1hZ2VzIiwgInZhbCI6ImltYWdlcyIsICJuYW1lcyI6bWFuaWZlc3RbIm5hbWVzIl19KSkKICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbiwgWU9MT19DT05GSUdfRElSPXN0cihvdXRwdXQvInlvbG9fc2V0dGluZ3MiKSwgV0FOREJfTU9ERT0iZGlzYWJsZWQiLCBDT01FVF9NT0RFPSJESVNBQkxFRCIpCiAgICBjb21tYW5kID0gW3N5cy5leGVjdXRhYmxlLCItbSIsInJldmFtcC5zbW9rZSIsIi0tcGFja2FnZSIsc3RyKHBhY2thZ2UpLCItLXdvcmtlciIsIi0tZGF0YSIsc3RyKGNmZyksIi0tZGV2aWNlIixzdHIoZGV2aWNlKV0KICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgaWYgbm90IGxvY2FsLmV4aXN0cygpOgogICAgICAgIGlmIGZpcnN0LmV4aXN0cygpOgogICAgICAgICAgICAjIEFuIGludGVycnVwdGlvbiBiZWZvcmUgdGhlIGZpcnN0IGNoZWNrcG9pbnQgaGFzIG5vIGVwb2NoIHRvIHJlc3VtZS4KICAgICAgICAgICAgIyBQcmVzZXJ2ZSB0aGF0IHBhcnRpYWwgZGlyZWN0b3J5IGluc3RlYWQgb2Ygb3ZlcndyaXRpbmcgaXRzIGZpbGVzLgogICAgICAgICAgICBmaXJzdC5yZW5hbWUob3V0cHV0L2YiaW5jb21wbGV0ZV9maXJzdF9ydW5fe3RpbWUudGltZV9ucygpfSIpCiAgICAgICAgcHJpbnQoIlN0YXJ0aW5nIHRpbnkgY2hlY2s7IHRoZSBmaXJzdCBjaGVja3BvaW50IHdpbGwgYmUgc2F2ZWQgYWZ0ZXIgZXBvY2ggMS4iLCBmbHVzaD1UcnVlKQogICAgICAgIHdpdGggKG91dHB1dC8icGF1c2UubG9nIikub3BlbigiYSIpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3VicHJvY2Vzcy5ydW4oY29tbWFuZCtbIi0tb3V0cHV0IixzdHIoZmlyc3QpXSwgZW52PWVudixzdGRvdXQ9c3RyZWFtLHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCxjaGVjaz1UcnVlKQogICAgICAgIGxvY2FsID0gY2suY2hlY2twb2ludF9wYXRoKGZpcnN0KQogICAgY2sucmVzdG9yZV9jaGVja3BvaW50KGxvY2FsLGZyZXNoLGV4cGVjdGVkKQogICAgaWYgbG9jYWwgIT0gY2suY2hlY2twb2ludF9wYXRoKGZyZXNoKToKICAgICAgICBzaHV0aWwuY29weWZpbGUobG9jYWwsY2suY2hlY2twb2ludF9wYXRoKGZyZXNoKSkKICAgIGlmIGZpcnN0LmV4aXN0cygpOgogICAgICAgIGZpcnN0LnJlbmFtZShvdXRwdXQvZiJwcmVzZXJ2ZWRfZmlyc3RfcnVuX3t0aW1lLnRpbWVfbnMoKX0iKQogICAgcHJpbnQoIkNvbnRpbnVpbmcgZnJvbSB0aGUgbGFzdCBjb21wbGV0ZSBjaGVja3BvaW50LiBQcm9ncmVzczogIiArIHN0cihvdXRwdXQvInJlc3VtZS5sb2ciKSxmbHVzaD1UcnVlKQogICAgd2l0aCAob3V0cHV0LyJyZXN1bWUubG9nIikub3BlbigiYSIpIGFzIHN0cmVhbToKICAgICAgICBzdWJwcm9jZXNzLnJ1bihjb21tYW5kK1siLS1vdXRwdXQiLHN0cihmcmVzaCksIi0tcmVzdW1lIl0sZW52PWVudixzdGRvdXQ9c3RyZWFtLHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCxjaGVjaz1UcnVlKQogICAgd2l0aCAoZnJlc2gvInRyYWluaW5nL3Jlc3VsdHMuY3N2Iikub3BlbigpIGFzIHN0cmVhbToKICAgICAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKSkKICAgIGVwb2NocyA9IFtpbnQoZmxvYXQocm93WyJlcG9jaCJdKSkgZm9yIHJvdyBpbiByb3dzXQogICAgaWYgZXBvY2hzICE9IGxpc3QocmFuZ2UoMSxSRUNJUEVbImVwb2NocyJdKzEpKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZWQgQ1NWIGhhcyBtaXNzaW5nIG9yIHJlcGVhdGVkIGVwb2Nocy4iKQogICAgZmlyc3RfbG9zcyA9IHN1bShmbG9hdChyWyJ0cmFpbi9ib3hfbG9zcyJdKSBmb3IgciBpbiByb3dzWzo1XSkvNQogICAgbGFzdF9sb3NzID0gc3VtKGZsb2F0KHJbInRyYWluL2JveF9sb3NzIl0pIGZvciByIGluIHJvd3NbLTU6XSkvNQogICAgYXA1MCA9IG1heChmbG9hdChyWyJtZXRyaWNzL21BUDUwKEIpIl0pIGZvciByIGluIHJvd3MpCiAgICBsZWFybmVkID0gbGFzdF9sb3NzIDwgLjg1KmZpcnN0X2xvc3MgYW5kIGFwNTAgPj0gLjUKICAgIHJlcG9ydCA9IHsic3RhdHVzIjoicGFzc2VkIiBpZiBsZWFybmVkIGVsc2UgImZhaWxlZF9sZWFybmluZ19nYXRlIiwgInNlY29uZHMiOnJvdW5kKHRpbWUubW9ub3RvbmljKCktc3RhcnQsMSksCiAgICAgICAgICAgICAgInNjb3BlIjoiU2l4IFRSQUlOSU5HIGNyb3BzIGV2YWx1YXRlZCBvbiB0aGVtc2VsdmVzLiBGdW5jdGlvbmFsIGV2aWRlbmNlIE9OTFk7IG5vdCBtb2RlbCBhY2N1cmFjeS4iLAogICAgICAgICAgICAgICJyZWNpcGUiOlJFQ0lQRSwgInBhY2thZ2VfaGFzaGVzX3NoYTI1NiI6ZGlnZXN0KHBhY2thZ2UvInBhY2thZ2VfaGFzaGVzLmpzb24iKSwKICAgICAgICAgICAgICAic2VsZWN0ZWRfcGFyZW50X2lkcyI6W3JbInBhcmVudF9pZCJdIGZvciByIGluIHNlbGVjdGVkLnZhbHVlcygpXSwKICAgICAgICAgICAgICAiY2hlY2tzIjpbIlJlYWwgdHJhaW5pbmcgcGF1c2VkIGFmdGVyIG9uZSBjb21wbGV0ZWQgZXBvY2giLCAiT3B0aW1pemVyIHJldGFpbmVkIGluIGNoZWNrc3VtbWVkIFpJUCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJSZXN1bWUgaW4gc2VwYXJhdGUgUHl0aG9uIHByb2Nlc3MgYW5kIGZyZXNoIGZvbGRlciB3aXRoIG9yaWdpbmFsIHJ1biBtb3ZlZCBhd2F5IiwKICAgICAgICAgICAgICAgICAgICAgICAgIk9wdGltaXplciByZXN0b3JlZCBhdCBzYXZlZCBlcG9jaCAob3IgZmluYWwgY2hlY2twb2ludCByZXVzZWQpIiwgIkNTViBoYXMgZWFjaCBwbGFubmVkIGVwb2NoIGV4YWN0bHkgb25jZSJdLAogICAgICAgICAgICAgICJsZWFybmluZ19nYXRlIjp7ImZpcnN0XzVfYm94X2xvc3MiOmZpcnN0X2xvc3MsImxhc3RfNV9ib3hfbG9zcyI6bGFzdF9sb3NzLCJiZXN0X3RyYWluX3NhbXBsZV9hcDUwIjphcDUwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJlcXVpcmVkIjoiTGFzdC1maXZlIG1lYW4gYm94IGxvc3MgPDg1JSBvZiBmaXJzdC1maXZlOyBtZW1vcml6ZWQtc2FtcGxlIEFQNTAgPj0wLjUwIn19CiAgICByZXBvcnRfdG1wID0gb3V0cHV0LyJyZXBvcnQudG1wIgogICAgcmVwb3J0X3RtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yKSsiXG4iKQogICAgcmVwb3J0X3RtcC5yZXBsYWNlKG91dHB1dC8icmVwb3J0Lmpzb24iKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIpKQogICAgaWYgbm90IGxlYXJuZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaW55IGxlYXJuaW5nIGNoZWNrIGZhaWxlZC4gSW5zcGVjdCBsb2dzOyBkbyBub3Qgc3RhcnQgZnVsbCB0cmFpbmluZy4iKQogICAgcmV0dXJuIHJlcG9ydAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wYWNrYWdlIix0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0Iix0eXBlPVBhdGgscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIixkZWZhdWx0PSJjcHUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrZXIiLGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VtZSIsYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YSIsdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50Iix0eXBlPVBhdGgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VtZS1vbmx5IixhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGlmIGFyZ3Mud29ya2VyOgogICAgICAgIHdvcmtlcihhcmdzLnBhY2thZ2UsYXJncy5kYXRhLGFyZ3Mub3V0cHV0LGFyZ3MucmVzdW1lLGFyZ3MuZGV2aWNlKQogICAgZWxzZToKICAgICAgICBjaGVjayhhcmdzLnBhY2thZ2UsYXJncy5vdXRwdXQsYXJncy5kZXZpY2UsYXJncy5jaGVja3BvaW50LGFyZ3MucmVzdW1lX29ubHkpCg==', 'sha256': '4579b39a6afba6191efa1599716bcdd9667aba987becad80c04e6a8d11ff65ae'}, 'pilot.py': {'base64': 'IiIiQm91bmRlZCBzcGVjaWFsaXN0IHBpbG90OiBoZWxkLW91dCB2YWxpZGF0aW9uLCBiYXNlbGluZSBjb21wYXJpc29uLCBlcG9jaCByZWNvdmVyeS4iIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCB0aW1lCgpmcm9tIHJldmFtcCBpbXBvcnQgY2hlY2twb2ludHMgYXMgY2sKZnJvbSByZXZhbXAucnVudGltZSBpbXBvcnQgTkFNRVMsIGNvbmZpZ3VyZSwgZGlnZXN0LCB2ZXJpZnlfcGFja2FnZQpmcm9tIHJldmFtcC5zbW9rZSBpbXBvcnQgY2hlY2twb2ludF9hcmNoaXZlCgoKZGVmIHJlY2lwZShkb21haW4pOgogICAgIyBGdWxsIHByZWNpc2lvbiBhdm9pZHMgaW50cm9kdWNpbmcgYW4gQU1QIHNjYWxlciB0aGF0IG91ciBjaGVja3BvaW50IGhlbHBlcgogICAgIyBkb2VzIG5vdCB5ZXQgcGVyc2lzdC4gTm8gc3RvY2hhc3RpYyB0cmFuc2Zvcm1zIGR1cmluZyB0aGUgdGltaW5nIHBpbG90LgogICAgcmV0dXJuIGRpY3QoZXBvY2hzPTMsIHBhdGllbmNlPTEwLCBpbWdzej05NjAgaWYgZG9tYWluID09ICdzb3VyY2UnIGVsc2UgNjQwLAogICAgICAgICAgICAgICAgYmF0Y2g9NCBpZiBkb21haW4gPT0gJ3NvdXJjZScgZWxzZSA4LCB3b3JrZXJzPTAsIG9wdGltaXplcj0nQWRhbVcnLAogICAgICAgICAgICAgICAgbHIwPS4wMDA1LCBscmY9LjEsIG5icz00IGlmIGRvbWFpbiA9PSAnc291cmNlJyBlbHNlIDgsCiAgICAgICAgICAgICAgICBjb3NfbHI9VHJ1ZSwgd2FybXVwX2Vwb2Nocz0wLCBzZWVkPTQyLCBkZXRlcm1pbmlzdGljPVRydWUsCiAgICAgICAgICAgICAgICBhbXA9RmFsc2UsIG1vc2FpYz0wLiwgbWl4dXA9MC4sIGNvcHlfcGFzdGU9MC4sIGN1dG1peD0wLiwKICAgICAgICAgICAgICAgIGRlZ3JlZXM9MC4sIHRyYW5zbGF0ZT0wLiwgc2NhbGU9MC4sIHNoZWFyPTAuLCBwZXJzcGVjdGl2ZT0wLiwKICAgICAgICAgICAgICAgIGZsaXBscj0wLiwgZmxpcHVkPTAuLCBoc3ZfaD0wLiwgaHN2X3M9MC4sIGhzdl92PTAuLCBiZ3I9MC4sCiAgICAgICAgICAgICAgICBhdWdtZW50YXRpb25zPVtdLCBjbG9zZV9tb3NhaWM9MCwgc2F2ZT1UcnVlLCBzYXZlX3BlcmlvZD0tMSwKICAgICAgICAgICAgICAgIHBsb3RzPUZhbHNlLCBjYWNoZT1GYWxzZSkKCgpkZWYgZW52aXJvbm1lbnQoKToKICAgIGltcG9ydCBwbGF0Zm9ybQogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdG9yY2h2aXNpb24KICAgIGltcG9ydCB1bHRyYWx5dGljcwogICAgaW1wb3J0IG51bXB5CiAgICByZXR1cm4gZGljdChweXRob249cGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwgdG9yY2g9dG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICB0b3JjaHZpc2lvbj10b3JjaHZpc2lvbi5fX3ZlcnNpb25fXywgdWx0cmFseXRpY3M9dWx0cmFseXRpY3MuX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICBudW1weT1udW1weS5fX3ZlcnNpb25fXywgY3VkYT10b3JjaC52ZXJzaW9uLmN1ZGEpCgoKZGVmIGlkZW50aXR5KHBhY2thZ2UsIGRvbWFpbik6CiAgICByZWNvcmQgPSBkaWN0KHBhY2thZ2Vfc2hhMjU2PWRpZ2VzdChQYXRoKHBhY2thZ2UpLydwYWNrYWdlX2hhc2hlcy5qc29uJyksCiAgICAgICAgICAgICAgICAgIGRvbWFpbj1kb21haW4sIHJlY2lwZT1yZWNpcGUoZG9tYWluKSwgaW5pdGlhbGl6YXRpb249J3dlaWdodHMveW9sb3Y4cy5wdCcsCiAgICAgICAgICAgICAgICAgIGVudmlyb25tZW50PWVudmlyb25tZW50KCksIGNvZGU9e25hbWU6ZGlnZXN0KFBhdGgoX19maWxlX18pLndpdGhfbmFtZShuYW1lKSkKICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiBbJ3BpbG90LnB5JywnY2hlY2twb2ludHMucHknLCdydW50aW1lLnB5Jywnc21va2UucHknXX0pCiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhyZWNvcmQsc29ydF9rZXlzPVRydWUpLmVuY29kZSgpKS5oZXhkaWdlc3QoKSwgcmVjb3JkCgoKZGVmIGF0b21pY19qc29uKHBhdGgsIHJlY29yZCk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeCgnLnRtcCcpCiAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpCiAgICB0bXAucmVwbGFjZShwYXRoKQoKCmRlZiBtZXRyaWNzX3JlY29yZChyZXN1bHQpOgogICAgYm94ID0gcmVzdWx0LmJveAogICAgcmV0dXJuIGRpY3QobWFwNTA9ZmxvYXQoYm94Lm1hcDUwKSwgbWFwNTBfOTU9ZmxvYXQoYm94Lm1hcCksCiAgICAgICAgICAgICAgICBwcmVjaXNpb249ZmxvYXQoYm94Lm1wKSwgcmVjYWxsPWZsb2F0KGJveC5tciksCiAgICAgICAgICAgICAgICBwZXJfY2xhc3M9e05BTUVTW2ludChjKV06ZGljdChhcDUwPWZsb2F0KGJveC5hcDUwW2ldKSwKICAgICAgICAgICAgICAgICAgICBhcDUwXzk1PWZsb2F0KGJveC5hcFtpXSksIHByZWNpc2lvbj1mbG9hdChib3gucFtpXSksIHJlY2FsbD1mbG9hdChib3gucltpXSkpCiAgICAgICAgICAgICAgICAgICAgZm9yIGksYyBpbiBlbnVtZXJhdGUoYm94LmFwX2NsYXNzX2luZGV4KX0sCiAgICAgICAgICAgICAgICBzcGVlZF9tc19wZXJfaW1hZ2U9e2s6ZmxvYXQodikgZm9yIGssdiBpbiByZXN1bHQuc3BlZWQuaXRlbXMoKX0pCgoKZGVmIGV2YWx1YXRlKHdlaWdodHMsIGRhdGEsIHJ1biwgbmFtZSwgc2V0dGluZ3MsIGRldmljZSwgZXhwZWN0ZWQpOgogICAgZnJvbSB1bHRyYWx5dGljcyBpbXBvcnQgWU9MTwogICAgcGF0aCA9IHJ1bi8obmFtZSsnLmpzb24nKQogICAga2V5ID0gZGljdChydW5fc2lnbmF0dXJlPWV4cGVjdGVkLCB3ZWlnaHRzX3NoYTI1Nj1kaWdlc3Qod2VpZ2h0cyksIGltZ3N6PXNldHRpbmdzWydpbWdzeiddLAogICAgICAgICAgICAgICBiYXRjaD1zZXR0aW5nc1snYmF0Y2gnXSwgY29uZj0uMDAxLCBpb3U9LjcsIG1heF9kZXQ9MzAwLCBzcGxpdD0ndmFsJykKICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmVjb3JkID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHJlY29yZFsnaWRlbnRpdHknXSAhPSBrZXk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1NhdmVkIGV2YWx1YXRpb24gZGlmZmVyczogJytzdHIocGF0aCkpCiAgICAgICAgcmV0dXJuIHJlY29yZAogICAgc3RhcnQ9dGltZS5tb25vdG9uaWMoKQogICAgbW9kZWw9WU9MTyhzdHIod2VpZ2h0cykpCiAgICBpZiBsaXN0KG1vZGVsLm5hbWVzLnZhbHVlcygpKSAhPSBOQU1FUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCYXNlbGluZS9jYW5kaWRhdGUgY2xhc3Mgb3JkZXIgZGlmZmVycy4nKQogICAgcmVzdWx0PW1vZGVsLnZhbChkYXRhPXN0cihkYXRhKSxzcGxpdD0ndmFsJyxpbWdzej1zZXR0aW5nc1snaW1nc3onXSxiYXRjaD1zZXR0aW5nc1snYmF0Y2gnXSwKICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSx3b3JrZXJzPTAsaGFsZj1GYWxzZSxjb25mPS4wMDEsaW91PS43LG1heF9kZXQ9MzAwLAogICAgICAgICAgICAgICAgICAgICBhdWdtZW50PUZhbHNlLHBsb3RzPUZhbHNlLHNhdmVfanNvbj1GYWxzZSxwcm9qZWN0PXN0cihydW4vJ2V2YWx1YXRpb24nKSwKICAgICAgICAgICAgICAgICAgICAgbmFtZT1uYW1lLGV4aXN0X29rPVRydWUsdmVyYm9zZT1UcnVlKQogICAgcmVjb3JkPWRpY3QoaWRlbnRpdHk9a2V5LG1ldHJpY3M9bWV0cmljc19yZWNvcmQocmVzdWx0KSxzZWNvbmRzPXRpbWUubW9ub3RvbmljKCktc3RhcnQpCiAgICBhdG9taWNfanNvbihwYXRoLHJlY29yZCkKICAgIHJldHVybiByZWNvcmQKCgpkZWYgYXNzZXJ0X25vX2FsYnVtZW50YXRpb25zKGRhdGFzZXQpOgogICAgIiIiRmFpbCBpZiB0aGUgaW5zdGFsbGVkIHRyYWluZXIgc2lsZW50bHkgZW5hYmxlcyBpdHMgb3B0aW9uYWwgZGVmYXVsdCB0cmFuc2Zvcm1zLiIiIgogICAgZGVmIHZpc2l0KHRyYW5zZm9ybSk6CiAgICAgICAgaWYgdHlwZSh0cmFuc2Zvcm0pLl9fbmFtZV9fID09ICdBbGJ1bWVudGF0aW9ucyc6CiAgICAgICAgICAgIGFwcGxpZWQ9Z2V0YXR0cih0cmFuc2Zvcm0sJ3RyYW5zZm9ybScsTm9uZSkKICAgICAgICAgICAgaWYgYXBwbGllZCBpcyBub3QgTm9uZSBhbmQgZ2V0YXR0cihhcHBsaWVkLCd0cmFuc2Zvcm1zJyxbXSk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgQWxidW1lbnRhdGlvbnMgdHJhbnNmb3JtcyBlbmFibGVkLicpCiAgICAgICAgZm9yIGNoaWxkIGluIGdldGF0dHIodHJhbnNmb3JtLCd0cmFuc2Zvcm1zJyxbXSk6CiAgICAgICAgICAgIHZpc2l0KGNoaWxkKQogICAgdmlzaXQoZGF0YXNldC50cmFuc2Zvcm1zKQoKCmRlZiB0cmFpbihwYWNrYWdlLCBkYXRhLCBydW4sIGRvbWFpbiwgZGV2aWNlLCBleHBlY3RlZCwgcmVzdW1lKToKICAgIGltcG9ydCB0b3JjaAogICAgZnJvbSB1bHRyYWx5dGljcyBpbXBvcnQgWU9MTwogICAgc2V0dGluZ3M9cmVjaXBlKGRvbWFpbikKICAgIGNvbXBsZXRlZD0wCiAgICBpZiByZXN1bWU6CiAgICAgICAgcmF3PXRvcmNoLmxvYWQocnVuLyd0cmFpbmluZy93ZWlnaHRzL2xhc3QucHQnLG1hcF9sb2NhdGlvbj0nY3B1Jyx3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgY29tcGxldGVkPWludChyYXdbJ2Vwb2NoJ10pKzEKICAgICAgICBpZiBjb21wbGV0ZWQgPCAxIG9yIHJhd1snb3B0aW1pemVyJ10gaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ2hlY2twb2ludCBoYXMgbm8gY29tcGxldGVkIGVwb2NoL29wdGltaXplci4nKQogICAgICAgIGRlbCByYXcKICAgICAgICBjay50cmltX3Jlc3VsdHMocnVuLGNvbXBsZXRlZCkKICAgICAgICBpZiBjb21wbGV0ZWQgPj0gc2V0dGluZ3NbJ2Vwb2NocyddOgogICAgICAgICAgICBwcmludCgnQWxsIHBpbG90IGVwb2NocyBzYXZlZDsgZXZhbHVhdGlvbi9yZXBvcnQgcmVjb3Zlcnkgb25seS4nLGZsdXNoPVRydWUpCiAgICAgICAgICAgIHJldHVybgogICAgbW9kZWw9WU9MTyhzdHIocnVuLyd0cmFpbmluZy93ZWlnaHRzL2xhc3QucHQnIGlmIHJlc3VtZSBlbHNlIHBhY2thZ2UvJ3dlaWdodHMveW9sb3Y4cy5wdCcpKQogICAgc3RhcnRlZD1bTm9uZV0KICAgIGRlZiBvbl9zdGFydCh0cmFpbmVyKToKICAgICAgICBhc3NlcnRfbm9fYWxidW1lbnRhdGlvbnModHJhaW5lci50cmFpbl9sb2FkZXIuZGF0YXNldCkKICAgICAgICBjay5yZXN0b3JlX3RyYWluZXJfc3RhdGUodHJhaW5lcixydW4pCiAgICAgICAgaWYgcmVzdW1lIGFuZCAodHJhaW5lci5zdGFydF9lcG9jaCAhPSBjb21wbGV0ZWQgb3Igbm90IHRyYWluZXIub3B0aW1pemVyLnN0YXRlKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPcHRpbWl6ZXIvZXBvY2ggcmVjb3ZlcnkgZmFpbGVkLicpCiAgICAgICAgcHJpbnQoZidQSUxPVCBSRUFEWTogZXBvY2gge3RyYWluZXIuc3RhcnRfZXBvY2grMX07IG9wdGlvbmFsIGF1Z21lbnRhdGlvbnMgZGlzYWJsZWQnLGZsdXNoPVRydWUpCiAgICBkZWYgZXBvY2hfc3RhcnQodHJhaW5lcik6CiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKTp0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgICAgICBzdGFydGVkWzBdPXRpbWUubW9ub3RvbmljKCkKICAgIGRlZiBzYXZlKHRyYWluZXIpOgogICAgICAgIHJlY29yZD1qc29uLmxvYWRzKChydW4vJ3NpZ25hdHVyZS5qc29uJykucmVhZF90ZXh0KCkpCiAgICAgICAgcmVjb3JkLnNldGRlZmF1bHQoJ2Vwb2NoX21lYXN1cmVtZW50cycsW10pLmFwcGVuZChkaWN0KGVwb2NoPXRyYWluZXIuZXBvY2grMSwKICAgICAgICAgICAgdHJhaW5fYW5kX3ZhbF9zZWNvbmRzPXRpbWUubW9ub3RvbmljKCktc3RhcnRlZFswXSwKICAgICAgICAgICAgcGVha19hbGxvY2F0ZWRfZ2liPXRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKS8xMDI0KiozIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBOb25lLAogICAgICAgICAgICBwZWFrX3Jlc2VydmVkX2dpYj10b3JjaC5jdWRhLm1heF9tZW1vcnlfcmVzZXJ2ZWQoKS8xMDI0KiozIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBOb25lKSkKICAgICAgICBhdG9taWNfanNvbihydW4vJ3NpZ25hdHVyZS5qc29uJyxyZWNvcmQpCiAgICAgICAgY2suc2F2ZV9jaGVja3BvaW50KHRyYWluZXIscnVuKQogICAgbW9kZWwuYWRkX2NhbGxiYWNrKCdvbl90cmFpbl9zdGFydCcsb25fc3RhcnQpCiAgICBtb2RlbC5hZGRfY2FsbGJhY2soJ29uX3RyYWluX2Vwb2NoX3N0YXJ0JyxlcG9jaF9zdGFydCkKICAgIG1vZGVsLmFkZF9jYWxsYmFjaygnb25fbW9kZWxfc2F2ZScsc2F2ZSkKICAgIHRyeToKICAgICAgICBpZiByZXN1bWU6CiAgICAgICAgICAgIG1vZGVsLnRyYWluKHJlc3VtZT1UcnVlLGRldmljZT1kZXZpY2UsZGF0YT1zdHIoZGF0YSksc2F2ZV9kaXI9c3RyKHJ1bi8ndHJhaW5pbmcnKSxhdWdtZW50YXRpb25zPVtdKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG1vZGVsLnRyYWluKGRhdGE9c3RyKGRhdGEpLGRldmljZT1kZXZpY2UscHJvamVjdD1zdHIocnVuKSxuYW1lPSd0cmFpbmluZycsZXhpc3Rfb2s9VHJ1ZSwqKnNldHRpbmdzKQogICAgZmluYWxseToKICAgICAgICB0cmFpbmVyPWdldGF0dHIobW9kZWwsJ3RyYWluZXInLE5vbmUpCiAgICAgICAgZm9yIGxvYWRlciBpbiBbZ2V0YXR0cih0cmFpbmVyLCd0cmFpbl9sb2FkZXInLE5vbmUpLGdldGF0dHIodHJhaW5lciwndGVzdF9sb2FkZXInLE5vbmUpXToKICAgICAgICAgICAgaWYgY2FsbGFibGUoZ2V0YXR0cihsb2FkZXIsJ2Nsb3NlJyxOb25lKSk6bG9hZGVyLmNsb3NlKCkKCgpkZWYgcnVuKHBhY2thZ2UsIG91dHB1dCwgZG9tYWluPSdzb3VyY2UnLCBkZXZpY2U9JzAnLCBjaGVja3BvaW50PU5vbmUsIHJlc3VtZV9vbmx5PUZhbHNlKToKICAgIGltcG9ydCBmY250bAogICAgcGFja2FnZSxvdXRwdXQ9UGF0aChwYWNrYWdlKS5yZXNvbHZlKCksUGF0aChvdXRwdXQpLnJlc29sdmUoKQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgd2l0aCAob3V0cHV0LydydW4ubG9jaycpLm9wZW4oJ2EnKSBhcyBsb2NrOgogICAgICAgIHRyeTpmY250bC5mbG9jayhsb2NrLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vycm9yIGFzIGVycm9yOnJhaXNlIFJ1bnRpbWVFcnJvcignUGlsb3QgYWxyZWFkeSBhY3RpdmUuJykgZnJvbSBlcnJvcgogICAgICAgIHJldHVybiBfcnVuKHBhY2thZ2Usb3V0cHV0LGRvbWFpbixkZXZpY2UsY2hlY2twb2ludCxyZXN1bWVfb25seSkKCgpkZWYgX3J1bihwYWNrYWdlLG91dHB1dCxkb21haW4sZGV2aWNlLGNoZWNrcG9pbnQscmVzdW1lX29ubHkpOgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdWx0cmFseXRpY3MKICAgIGlmIHVsdHJhbHl0aWNzLl9fdmVyc2lvbl9fICE9ICc4LjQuMTQyJzpyYWlzZSBWYWx1ZUVycm9yKCdVc2UgcGlubmVkIFVsdHJhbHl0aWNzIDguNC4xNDIuJykKICAgIGlmIGRldmljZSAhPSAnY3B1JyBhbmQgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6cmFpc2UgUnVudGltZUVycm9yKCdFbmFibGUgS2FnZ2xlIEdQVSBiZWZvcmUgdGhlIHBpbG90LicpCiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCkKICAgIHZlcmlmeV9wYWNrYWdlKHBhY2thZ2UpCiAgICBleHBlY3RlZCxkZXRhaWxzPWlkZW50aXR5KHBhY2thZ2UsZG9tYWluKQogICAgcnVuPW91dHB1dC8ncGlsb3RfcnVuJzthcmNoaXZlPWNrLmNoZWNrcG9pbnRfcGF0aChydW4pCiAgICBpZiBjaGVja3BvaW50IGFuZCBhcmNoaXZlLmV4aXN0cygpOnJhaXNlIFZhbHVlRXJyb3IoJ0xvY2FsIHByb2dyZXNzIGV4aXN0czsgY2xlYXIgUkVTVE9SRV9GUk9NLicpCiAgICBpZiByZXN1bWVfb25seSBhbmQgbm90IGNoZWNrcG9pbnQgYW5kIG5vdCBhcmNoaXZlLmV4aXN0cygpOnJhaXNlIFJ1bnRpbWVFcnJvcignUmVzdW1lIHJlcXVlc3RlZCBidXQgbm8gY2hlY2twb2ludCBleGlzdHMuJykKICAgIG1hcmtlcj1ydW4vJ3NpZ25hdHVyZS5qc29uJwogICAgaWYgbWFya2VyLmV4aXN0cygpIGFuZCBqc29uLmxvYWRzKG1hcmtlci5yZWFkX3RleHQoKSlbJ3NoYTI1NiddICE9IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0RpZmZlcmVudCBwYWNrYWdlLCByZWNpcGUsIGNvZGUgb3IgZW52aXJvbm1lbnQ7IHVzZSBhIG5ldyBvdXRwdXQgZm9sZGVyLicpCiAgICBpZiBjaGVja3BvaW50OgogICAgICAgIGltcG9ydGVkPWNoZWNrcG9pbnRfYXJjaGl2ZShjaGVja3BvaW50LG91dHB1dCkKICAgICAgICBjay5yZXN0b3JlX2NoZWNrcG9pbnQoaW1wb3J0ZWQscnVuLGV4cGVjdGVkKQogICAgICAgIHNodXRpbC5jb3B5ZmlsZShpbXBvcnRlZCxhcmNoaXZlKQogICAgaWYgYXJjaGl2ZS5leGlzdHMoKTpjay5yZXN0b3JlX2NoZWNrcG9pbnQoYXJjaGl2ZSxydW4sZXhwZWN0ZWQpCiAgICBpZiBub3QgbWFya2VyLmV4aXN0cygpOgogICAgICAgIHJ1bi5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBhdG9taWNfanNvbihtYXJrZXIsZGljdChzaGEyNTY9ZXhwZWN0ZWQsaWRlbnRpdHk9ZGV0YWlscyxlcG9jaF9tZWFzdXJlbWVudHM9W10pKQogICAgcmVwb3J0X3BhdGg9b3V0cHV0LydyZXBvcnQuanNvbicKICAgIGlmIHJlcG9ydF9wYXRoLmV4aXN0cygpOgogICAgICAgIHJlcG9ydD1qc29uLmxvYWRzKHJlcG9ydF9wYXRoLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHJlcG9ydFsnc2lnbmF0dXJlJ10gIT0gZXhwZWN0ZWQgb3Igbm90IGFyY2hpdmUuZXhpc3RzKCkgb3IgcmVwb3J0WydjaGVja3BvaW50X3NoYTI1NiddICE9IGRpZ2VzdChhcmNoaXZlKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ29tcGxldGVkIHJlcG9ydC9jaGVja3BvaW50IG1pc21hdGNoLicpCiAgICAgICAgcHJpbnQoJ1BpbG90IGFscmVhZHkgY29tcGxldGU7IHJldXNpbmcgcmVwb3J0LicsZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gcmVwb3J0CiAgICBjb25maWdzPWNvbmZpZ3VyZShwYWNrYWdlLG91dHB1dC8nY29uZmlncycpCiAgICBkYXRhPWNvbmZpZ3NbJ3NvdXJjZV92aWV3cycgaWYgZG9tYWluPT0nc291cmNlJyBlbHNlICd0YXJnZXQnXQogICAgc2V0dGluZ3M9cmVjaXBlKGRvbWFpbikKICAgIGJhc2VsaW5lPWV2YWx1YXRlKHBhY2thZ2UvZid3ZWlnaHRzL3tkb21haW59X2Jhc2VsaW5lLnB0JyxkYXRhLHJ1biwKICAgICAgICAgICAgICAgICAgICAgIGYnYmFzZWxpbmVfe2RvbWFpbn1fdmFsaWRhdGlvbicsc2V0dGluZ3MsZGV2aWNlLGV4cGVjdGVkKQogICAgIyBJbnRlcnJ1cHRlZCBlcG9jaCBmaWxlcyBjYW5ub3Qgb3ZlcnJpZGUgdGhlIGF1dGhvcml0YXRpdmUgYXRvbWljIFpJUC4KICAgIGlmIG5vdCBhcmNoaXZlLmV4aXN0cygpIGFuZCAocnVuLyd0cmFpbmluZycpLmV4aXN0cygpOgogICAgICAgIChydW4vJ3RyYWluaW5nJykucmVuYW1lKHJ1bi9mJ2luY29tcGxldGVfdHJhaW5pbmdfe3RpbWUudGltZV9ucygpfScpCiAgICB0cmFpbihwYWNrYWdlLGRhdGEscnVuLGRvbWFpbixkZXZpY2UsZXhwZWN0ZWQsYXJjaGl2ZS5leGlzdHMoKSkKICAgIGNrLnJlc3RvcmVfY2hlY2twb2ludChhcmNoaXZlLHJ1bixleHBlY3RlZCkKICAgIHdpdGggKHJ1bi8ndHJhaW5pbmcvcmVzdWx0cy5jc3YnKS5vcGVuKCkgYXMgc3RyZWFtOgogICAgICAgIHJvd3M9bGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0sc2tpcGluaXRpYWxzcGFjZT1UcnVlKSkKICAgIGlmIFtpbnQoZmxvYXQoclsnZXBvY2gnXSkpIGZvciByIGluIHJvd3NdICE9IGxpc3QocmFuZ2UoMSxzZXR0aW5nc1snZXBvY2hzJ10rMSkpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUGlsb3QgZXBvY2hzIGluY29tcGxldGUgb3IgZHVwbGljYXRlZC4nKQogICAgY2FuZGlkYXRlPWV2YWx1YXRlKHJ1bi8ndHJhaW5pbmcvd2VpZ2h0cy9iZXN0LnB0JyxkYXRhLHJ1biwnY2FuZGlkYXRlX3ZhbGlkYXRpb24nLHNldHRpbmdzLGRldmljZSxleHBlY3RlZCkKICAgIG1lYXN1cmVtZW50cz1qc29uLmxvYWRzKG1hcmtlci5yZWFkX3RleHQoKSlbJ2Vwb2NoX21lYXN1cmVtZW50cyddCiAgICBpZiBbclsnZXBvY2gnXSBmb3IgciBpbiBtZWFzdXJlbWVudHNdICE9IGxpc3QocmFuZ2UoMSxzZXR0aW5nc1snZXBvY2hzJ10rMSkpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUGlsb3QgdGltaW5nIGhpc3RvcnkgaW5jb21wbGV0ZSBvciBkdXBsaWNhdGVkLicpCiAgICBtYW5pZmVzdD1qc29uLmxvYWRzKChwYWNrYWdlLydtYW5pZmVzdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICBkb21haW5fcm93cz1bciBmb3IgciBpbiBtYW5pZmVzdFsndmlld3MnXSBpZiByWydkYXRhc2V0J109PWRvbWFpbl0KICAgIHN0ZWFkeT1zdGF0aXN0aWNzLm1lZGlhbihtWyd0cmFpbl9hbmRfdmFsX3NlY29uZHMnXSBmb3IgbSBpbiBtZWFzdXJlbWVudHNbMTpdIG9yIG1lYXN1cmVtZW50cykKICAgIHJlcG9ydD1kaWN0KHN0YXR1cz0ncGlsb3RfY29tcGxldGVkJyxzaWduYXR1cmU9ZXhwZWN0ZWQsY2hlY2twb2ludF9zaGEyNTY9ZGlnZXN0KGFyY2hpdmUpLAogICAgICAgICAgICAgICAgaWRlbnRpdHk9ZGV0YWlscyxkZXZpY2U9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgZGV2aWNlIT0nY3B1JyBlbHNlICdjcHUnLAogICAgICAgICAgICAgICAgY291bnRzPXtzOnN1bShyWydzcGxpdCddPT1zIGZvciByIGluIGRvbWFpbl9yb3dzKSBmb3IgcyBpbiBbJ3RyYWluJywndmFsJ119LAogICAgICAgICAgICAgICAgYmFzZWxpbmU9YmFzZWxpbmUsY2FuZGlkYXRlPWNhbmRpZGF0ZSwKICAgICAgICAgICAgICAgIGRlbHRhPXtrOmNhbmRpZGF0ZVsnbWV0cmljcyddW2tdLWJhc2VsaW5lWydtZXRyaWNzJ11ba10gZm9yIGsgaW4gWydtYXA1MCcsJ21hcDUwXzk1JywncHJlY2lzaW9uJywncmVjYWxsJ119LAogICAgICAgICAgICAgICAgZXBvY2hfbWVhc3VyZW1lbnRzPW1lYXN1cmVtZW50cyxzdGVhZHlfZXBvY2hfc2Vjb25kc19leGNsdWRpbmdfY2hlY2twb2ludD1zdGVhZHksCiAgICAgICAgICAgICAgICBsb3dlcl9ib3VuZF82MF9lcG9jaF9ob3Vycz1zdGVhZHkqNjAvMzYwMCwKICAgICAgICAgICAgICAgIGxpbWl0YXRpb25zPVsnVGhyZWUgZXBvY2hzIGFyZSBhIHBpcGVsaW5lL3RpbWluZyBwaWxvdCwgbm90IGEgZmluYWwgYWNjdXJhY3kgcmVzdWx0LicsCiAgICAgICAgICAgICAgICAgICAgJ0VzdGltYXRlIGV4Y2x1ZGVzIGNoZWNrcG9pbnQgWklQIHdyaXRpbmcsIHNldHVwLCBmaW5hbCBldmFsdWF0aW9uIGFuZCBwb3NzaWJsZSBsYXRlciByZWNpcGUgY2hhbmdlcy4nLAogICAgICAgICAgICAgICAgICAgICdWYWxpZGF0aW9uIGlzIHNlcGFyYXRlIGZyb20gdHJhaW5pbmc7IG5vIGRldmVsb3BtZW50LXRlc3QgaW1hZ2VzIHVzZWQuJywKICAgICAgICAgICAgICAgICAgICAnVGFyZ2V0IG1pc3NpbmctaG9sZSBwZXJmb3JtYW5jZSB1bnN1cHBvcnRlZDsgdGFyZ2V0IGJvYXJkIGluZGVwZW5kZW5jZSB1bmtub3duLicsCiAgICAgICAgICAgICAgICAgICAgJ05vIG5vcm1hbCBmdWxsLWJvYXJkIGJlbmNobWFyazsgbm8gcHJvZHVjdGlvbiBtb2RlbCByZXBsYWNlZC4nXSkKICAgIGF0b21pY19qc29uKHJlcG9ydF9wYXRoLHJlcG9ydCkKICAgIHByaW50KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yKSxmbHVzaD1UcnVlKQogICAgcmV0dXJuIHJlcG9ydAoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXI9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tcGFja2FnZScsdHlwZT1QYXRoLHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dHB1dCcsdHlwZT1QYXRoLHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWRvbWFpbicsY2hvaWNlcz1bJ3NvdXJjZScsJ3RhcmdldCddLGRlZmF1bHQ9J3NvdXJjZScpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWRldmljZScsZGVmYXVsdD0nMCcpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWNoZWNrcG9pbnQnLHR5cGU9UGF0aCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tcmVzdW1lLW9ubHknLGFjdGlvbj0nc3RvcmVfdHJ1ZScpCiAgICBhcmdzPXBhcnNlci5wYXJzZV9hcmdzKCkKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgnWU9MT19DT05GSUdfRElSJyxzdHIoYXJncy5vdXRwdXQvJ3lvbG9fc2V0dGluZ3MnKSkKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgnV0FOREJfTU9ERScsJ2Rpc2FibGVkJykKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgnQ09NRVRfTU9ERScsJ0RJU0FCTEVEJykKICAgIHJ1bihhcmdzLnBhY2thZ2UsYXJncy5vdXRwdXQsYXJncy5kb21haW4sYXJncy5kZXZpY2UsYXJncy5jaGVja3BvaW50LGFyZ3MucmVzdW1lX29ubHkpCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'sha256': '999ae97097074c1f57c68b1ff1e2352117e05b7b2eb030e8a4dc53f96cac8d99'}}

import base64, importlib
TOOLS = Path('/kaggle/working/pcb_pilot_tools')
(TOOLS/'revamp').mkdir(parents=True, exist_ok=True)
for name, record in EMBEDDED_TOOLS.items():
    raw = base64.b64decode(record['base64'])
    assert hashlib.sha256(raw).hexdigest() == record['sha256']
    destination = TOOLS/'revamp'/name
    if destination.exists() and destination.read_bytes() != raw:
        raise RuntimeError('Different resume code already installed; use a fresh kernel/session.')
    if not destination.exists(): destination.write_bytes(raw)
sys.path.insert(0, str(TOOLS))
# A previous notebook import can remain cached after rerunning a cell.
for name in list(sys.modules):
    if name == 'revamp' or name.startswith('revamp.'): del sys.modules[name]
importlib.invalidate_caches()
from revamp.runtime import verify_package, configure
print(verify_package(WORK))
CONFIGS = configure(WORK, Path('/kaggle/working/pcb_phase2_configs'))
print('Setup verified; existing matching files reused.', CONFIGS)


In [ ]:
import platform, ultralytics
# Import failures here should be resolved before training; never silently change torch.
from ultralytics import YOLO
environment = {'python':platform.python_version(), 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'ultralytics':ultralytics.__version__, 'cuda_available':torch.cuda.is_available(), 'cuda':torch.version.cuda}
Path('/kaggle/working/pcb_environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
for name in ['yolov8n', 'yolov8s']:
    assert len(YOLO(str(WORK/f'weights/{name}.pt')).names) == 80
for name in ['source', 'target']:
    assert list(YOLO(str(WORK/f'weights/{name}_baseline.pt')).names.values()) == json.loads((WORK/'manifest.json').read_text())['names']

assert torch.cuda.is_available(), 'Enable GPU T4 before running this pilot.'
environment['gpu'] = torch.cuda.get_device_name(0)
Path('/kaggle/working/pcb_environment.json').write_text(json.dumps(environment, indent=2))


In [ ]:
# Run source first. Review its report before scheduling the target pilot.
DOMAIN = 'source'
RESTORE_FROM = None  # Attached pilot_run_checkpoint.zip or expanded checkpoint folder
RESUME_ONLY = False  # True refuses a fresh start without a checkpoint
RUN = Path(f'/kaggle/working/pcb_phase3_{DOMAIN}_pilot_v1')
RUN.mkdir(parents=True, exist_ok=True)
import signal
from IPython.display import display, FileLink
command = [sys.executable, '-m', 'revamp.pilot', '--package', str(WORK),
           '--output', str(RUN), '--domain', DOMAIN, '--device', '0']
if RESTORE_FROM: command += ['--checkpoint', str(RESTORE_FROM)]
if RESUME_ONLY: command += ['--resume-only']
env = dict(os.environ, PYTHONPATH=str(TOOLS), MPLCONFIGDIR='/kaggle/working/pcb_mpl',
           YOLO_CONFIG_DIR=str(RUN/'yolo_settings'), WANDB_MODE='disabled', COMET_MODE='DISABLED')
print('Pilot output:', RUN)
print('Baseline validation runs first. Checkpoints start after training epoch 1.')
print('Interrupt stops the worker. Rerun to resume the last saved epoch.')
log = RUN/'pilot.log'
with log.open('a') as stream:
    process = subprocess.Popen(command, cwd=TOOLS, env=env, stdout=stream,
                               stderr=subprocess.STDOUT, start_new_session=True)
    try:
        while True:
            try:
                returncode = process.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                lines = log.read_text(errors='replace').splitlines()
                if lines: print(lines[-1][-500:])
        if returncode:
            print('\n'.join(log.read_text(errors='replace').splitlines()[-25:]))
            raise RuntimeError('Pilot stopped or failed. Inspect pilot.log; saved checkpoints remain available. For OOM, stop and review before changing the recipe.')
    except KeyboardInterrupt:
        try: os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError: pass
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        print('Stopped. Rerun to recover the last complete epoch.')
    finally:
        for path in [RUN/'pilot_run_checkpoint.zip', RUN/'report.json', log,
                     Path('/kaggle/working/pcb_environment.json')]:
            if path.exists(): display(FileLink(os.path.relpath(path, Path.cwd())))
if (RUN/'report.json').exists(): print((RUN/'report.json').read_text())


## Stop and review

Download **report.json**, **pilot.log**, **pcb_environment.json** and **pilot_run_checkpoint.zip**. Send the report and log for review before a longer training run. Low validation scores after three epochs do not mean the dataset or pipeline failed; do not select a model from this pilot alone.

Same session: leave RESTORE_FROM=None and rerun. New session: attach the original PCB dataset and your trusted pilot checkpoint; set RESTORE_FROM to the ZIP or expanded folder. Clear RESTORE_FROM on subsequent local reruns. A completed checkpoint rebuilds missing evaluation/report outputs without training again. Checkpoints from the six-image smoke test are incompatible and rejected.

Timing estimates exclude setup, checkpoint writing and final evaluation. Source and target are separate experiments. Target has five supported classes and cannot establish missing-hole performance. Do not change DOMAIN until the source pilot has been reviewed. No app model is replaced.
